# 01 - Career age event study subgroup

This notebook prepares a cleaner subgroup for the descriptive
event study analysis.

## 1. Setup

In [1]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "project_setup.py").exists() and (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError(
        "Could not find the repository root. Launch Jupyter from the repo root "
        "or set PYTHONPATH to the folder containing project_setup.py."
    )

import json
import os
import sys
import time
from datetime import date
from pathlib import Path

os.environ.setdefault("ARROW_USER_SIMD_LEVEL", "NONE")

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import IFrame, Markdown, display
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import font_manager
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from matplotlib.ticker import MaxNLocator
from matplotlib.transforms import blended_transform_factory
from PIL import Image, ImageDraw, ImageFilter, ImageFont

from author_matching import normalize_name
from project_setup import ensure_dirs, load_project_env, setup_project

setup = setup_project()
PROJECT = setup.project_folder
loaded_env_files = load_project_env(PROJECT)

STEP_2_PREPARED = PROJECT / "step_2_data" / "prepared" / "all_papers"
STEP_3_PREPARED = PROJECT / "step_3_data" / "prepared"
STEP_3_SUMMARY = PROJECT / "step_3_artifacts" / "summary_tables"

STEP_4_FIRST_WORK_RAW = (
    PROJECT / "step_4_data" / "raw" / "openalex_author_first_works"
)
STEP_4_FIRST_CS_WORK_RAW = (
    PROJECT / "step_4_data" / "raw" / "openalex_author_first_cs_works"
)
STEP_4_AUTHOR_PROFILE_RAW = (
    PROJECT / "step_4_data" / "raw" / "openalex_author_profiles"
)
STEP_4_PREPARED = PROJECT / "step_4_data" / "prepared"
STEP_4_SUMMARY = PROJECT / "step_4_artifacts" / "summary_tables"
MAIN_TEXT_FIGURES = PROJECT / "step_4_artifacts" / "main_text_figures"
PRE_PC_MEAN_BASELINE_FIGURES = (
    PROJECT / "step_4_artifacts" / "figures_pre_pc_mean_baseline_event_study"
)
REPORT_FIGURES = PROJECT / "report_latex" / "figures"
REPORT_GS_SCREENSHOT_DIR = REPORT_FIGURES / "google_scholar_screenshots"
ensure_dirs(
    STEP_4_FIRST_WORK_RAW,
    STEP_4_FIRST_CS_WORK_RAW,
    STEP_4_AUTHOR_PROFILE_RAW,
    STEP_4_PREPARED,
    STEP_4_SUMMARY,
    MAIN_TEXT_FIGURES,
    PRE_PC_MEAN_BASELINE_FIGURES,
    REPORT_FIGURES,
    REPORT_GS_SCREENSHOT_DIR,
)

PANEL_PATH = STEP_3_PREPARED / "panel.parquet"
PANEL_FEATURES_PATH = STEP_3_PREPARED / "panel_features.parquet"
CITATION_KEYS_PATH = STEP_3_PREPARED / "panel_citation_keys.parquet"
REF_AUTHORS_PATH = STEP_2_PREPARED / "all_ref_authors_exploded.parquet"
PAPERS_PATH = STEP_2_PREPARED / "all_papers_filtered.parquet"
IDENTITY_TABLE_PATH = STEP_3_SUMMARY / "pc_researcher_identity_validation.csv"

RELIABLE_AUTHOR_LIST_OUT = STEP_4_SUMMARY / "career_age_reliable_author_list.csv"
SUBGROUP_SUMMARY_OUT = STEP_4_SUMMARY / "career_age_subgroup_summary.csv"
FETCH_SUMMARY_OUT = STEP_4_SUMMARY / "first_publication_fetch.csv"
AUTHOR_METRICS_OUT = STEP_4_PREPARED / "researcher_openalex_author_metrics.parquet"
EVENT_ROWS_OUT = STEP_4_PREPARED / "career_age_event_window_rows.parquet"
EVENT_UNITS_OUT = STEP_4_SUMMARY / "career_age_event_unit_summary.csv"
EVENT_LOG_SUMMARY_OUT = STEP_4_SUMMARY / "career_age_event_study_log_summary.csv"
EVENT_RAW_SUMMARY_OUT = STEP_4_SUMMARY / "career_age_event_study_raw_summary.csv"
EVENT_LOG_BY_CONFERENCE_OUT = (
    STEP_4_SUMMARY / "career_age_log_by_conf.csv"
)
EVENT_RAW_BY_CONFERENCE_OUT = (
    STEP_4_SUMMARY / "career_age_raw_by_conf.csv"
)
MAIN_EVENT_LOG_SUMMARY_OUT = (
    STEP_4_SUMMARY / "icfp_selection_log.csv"
)
MAIN_EVENT_RAW_SUMMARY_OUT = (
    STEP_4_SUMMARY / "icfp_selection_raw.csv"
)
MAIN_EVENT_SELECTED_LARGE_JUMPS_OUT = (
    STEP_4_SUMMARY / "icfp_selected_jumps.csv"
)
MAIN_EVENT_SELECTED_CITATION_SOURCES_OUT = (
    STEP_4_SUMMARY / "icfp_selected_sources.csv"
)
MAIN_EVENT_SELECTED_PRIOR_WORK_SUMMARY_OUT = (
    STEP_4_SUMMARY / "icfp_selected_prior_summary.csv"
)
MAIN_EVENT_SELECTED_PRIOR_WORK_COMPACT_OUT = (
    STEP_4_SUMMARY / "icfp_selected_prior_work.csv"
)
MAIN_EVENT_SELECTED_CITING_PAPER_SUMMARY_OUT = (
    STEP_4_SUMMARY / "icfp_selected_citing_papers.csv"
)
MAIN_EVENT_LOG_FIGURE_OUT = PRE_PC_MEAN_BASELINE_FIGURES / "icfp_log.pdf"
MAIN_EVENT_RAW_FIGURE_OUT = PRE_PC_MEAN_BASELINE_FIGURES / "icfp_raw.pdf"
REPORT_MAIN_EVENT_LOG_FIGURE_OUT = REPORT_FIGURES / "pre_pc_icfp_log.pdf"
REPORT_MAIN_EVENT_RAW_FIGURE_OUT = REPORT_FIGURES / "pre_pc_icfp_raw.pdf"
SPIKE_FADE_CASES_OUT = STEP_4_SUMMARY / "selected_spike_and_fade_cases.csv"
SPIKE_FADE_T0_SOURCE_SUMMARY_OUT = (
    STEP_4_SUMMARY / "selected_t0_source_summary.csv"
)
SPIKE_FADE_T0_SOURCE_DETAIL_OUT = (
    STEP_4_SUMMARY / "selected_t0_source_detail.csv"
)
SELECTED_EVENT_STUDY_ANALYSIS_ROWS_OUT = (
    STEP_4_SUMMARY / "selected_pc_event_study_analysis_rows.csv"
)
SELECTED_EVENT_STUDY_ANALYSIS_MEAN_SUMMARY_OUT = (
    STEP_4_SUMMARY / "selected_event_means.csv"
)
SELECTED_EVENT_STUDY_ANALYSIS_FIGURE_OUT = (
    MAIN_TEXT_FIGURES / "selected_pc_event_study_analysis.pdf"
)
REPORT_SELECTED_EVENT_STUDY_ANALYSIS_FIGURE_OUT = (
    REPORT_FIGURES / "selected_pc_event_study_analysis.pdf"
)
SPIKE_FADE_FIGURE_OUT = MAIN_TEXT_FIGURES / "selected_pc_trajectories.pdf"
REPORT_SPIKE_FADE_FIGURE_OUT = (
    REPORT_FIGURES / "selected_pc_trajectories.pdf"
)
GS_SCREENSHOT_DIR = PROJECT / "assets" / "selected_researcher_career_age"
GS_NODE_DIR = GS_SCREENSHOT_DIR / "story_candidate_nodes"
GS_NODE_HIGH_RES_DIR = GS_SCREENSHOT_DIR / "story_candidate_nodes_hires"
STORY_NODE_SCALE = 4
STORY_NODE_EXPORT_DPI = 450
ensure_dirs(GS_NODE_HIGH_RES_DIR)
REPORT_GS_SCREENSHOT_GRID_OUT = (
    REPORT_FIGURES / "google_scholar_career_age_screenshots.png"
)
print("Project folder: .")
print(f"Run mode: {setup.run_mode}")
print(f"Allow network: {setup.allow_network}")
print(f"Overwrite data: {setup.overwrite_data}")
print(f"Overwrite artifacts: {setup.overwrite_artifacts}")
print(f"Loaded env files: {loaded_env_files}")
print(f"Run date: {date.today().isoformat()}")

Project folder: .
Run mode: fast
Allow network: False
Overwrite data: False
Overwrite artifacts: True
Loaded env files: []
Run date: 2026-06-19


## 2. Plot style

In [2]:
font_path = PROJECT / "fonts" / "LinLibertine_R.ttf"
if font_path.exists():
    font_manager.fontManager.addfont(str(font_path))
    font_prop = font_manager.FontProperties(fname=font_path)
    font_family = font_prop.get_name()
else:
    font_family = "serif"

mpl.rcParams.update(
    {
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "font.size": 11,
        "legend.fontsize": 10,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "font.family": font_family,
        "text.usetex": True,
        "axes.linewidth": 0.8,
        "axes.edgecolor": "black",
        "xtick.direction": "out",
        "ytick.direction": "out",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)

MAIN_COLOR = "#81c8be"
RAW_COLOR = "#e5c890"
CONFERENCE_COLORS = {
    "POPL": "#81c8be",
    "ICFP": "#f5bde6",
    "OOPSLA": "#8aadf4",
    "OOPSLA1": "#8aadf4",
    "OOPSLA2": "#8aadf4",
    "OOPSLA combined": "#8aadf4",
    "PLDI": "#e5c890",
    "PLDI before 2021": "#e5c890",
    "PLDI 2021 onward": "#e5c890",
}

## 3. Read the final panel and identity table

In [3]:
panel = pd.read_parquet(PANEL_PATH)
panel_features = pd.read_parquet(PANEL_FEATURES_PATH)
identity = pd.read_csv(IDENTITY_TABLE_PATH)

panel["year"] = panel["year"].astype(int)
panel["pc_status"] = panel["pc_status"].astype(int)
panel["citation_count"] = pd.to_numeric(panel["citation_count"], errors="coerce")

feature_cols = [
    col
    for col in panel_features.columns
    if col not in {"researcher_id", "conference", "year"}
]
panel_full = panel.merge(
    panel_features[feature_cols],
    on="panel_row_id",
    how="left",
    validate="one_to_one",
)

identity = identity.rename(columns={"canonical_researchr_id": "researcher_id"})
identity["ideal_openalex_case"] = (
    identity.get("ideal_openalex_case", False).fillna(False).astype(bool)
)

print(f"panel: {panel.shape}")
print(f"panel_features: {panel_features.shape}")
print(f"panel_full: {panel_full.shape}")
print(f"identity table: {identity.shape}")
display(panel_full.head())

panel: (37128, 7)
panel_features: (37128, 39)
panel_full: (37128, 42)
identity table: (952, 39)


,panel_row_id,researcher_id,name,conference,year,pc_status,citation_count,mapped_name_for_matching,openalex_id,orcid_from_openalex,...,source_verified_first_broad_service_year_any_target,had_prior_pc_before_observed_conf,had_prior_broad_service_before_observed_conf,prior_pc_evidence_count_conf,prior_broad_service_evidence_count_conf,is_true_first_pc_in_observed_year_conf,is_true_first_broad_service_in_observed_year_conf,citation_count_no_self,citation_count_first_author,citation_count_first_or_last
0,aaronbembenek|ICFP|2017,aaronbembenek,Aaron Bembenek,ICFP,2017,0,0.0,Aaron Bembenek,NaN,0000-0002-3677-701X,...,2025,None,None,NaN,NaN,None,None,0.0,0.0,0.0
1,aaronbembenek|ICFP|2018,aaronbembenek,Aaron Bembenek,ICFP,2018,0,0.0,Aaron Bembenek,NaN,0000-0002-3677-701X,...,2025,None,None,NaN,NaN,None,None,0.0,0.0,0.0
2,aaronbembenek|ICFP|2019,aaronbembenek,Aaron Bembenek,ICFP,2019,0,0.0,Aaron Bembenek,NaN,0000-0002-3677-701X,...,2025,None,None,NaN,NaN,None,None,0.0,0.0,0.0
3,aaronbembenek|ICFP|2020,aaronbembenek,Aaron Bembenek,ICFP,2020,0,0.0,Aaron Bembenek,NaN,0000-0002-3677-701X,...,2025,None,None,NaN,NaN,None,None,0.0,0.0,0.0
4,aaronbembenek|ICFP|2021,aaronbembenek,Aaron Bembenek,ICFP,2021,0,0.0,Aaron Bembenek,NaN,0000-0002-3677-701X,...,2025,None,None,NaN,NaN,None,None,0.0,0.0,0.0


## 4. Define the reliable career age subgroup

The full panel keeps all PC researchers, but career age requires
author level OpenAlex metadata. For the main career age check I
keep researchers who have:

- an OpenAlex Author ID;
- an identity confidence label that is not red.

I also keep stricter flags for sensitivity checks, such as ORCID
agreement and ideal one name, one author ID cases.

In [4]:
service_summary = (
    panel_full.groupby("researcher_id", as_index=False)
    .agg(
        first_pc_year_any=("first_pc_year_any", "min"),
        n_panel_rows=("panel_row_id", "size"),
        n_pc_service_cells=("pc_status", "sum"),
    )
)
served_conferences = (
    panel_full.loc[panel_full["pc_status"].eq(1)]
    .groupby("researcher_id")["conference"]
    .nunique()
    .reset_index(name="n_conferences_served")
)
service_summary = service_summary.merge(
    served_conferences,
    on="researcher_id",
    how="left",
)
service_summary["n_conferences_served"] = (
    service_summary["n_conferences_served"].fillna(0).astype(int)
)

researcher_table = identity.merge(service_summary, on="researcher_id", how="left")
researcher_table["use_for_main_career_age"] = (
    researcher_table["openalex_id"].notna()
    & researcher_table["match_confidence"].ne("RED")
)
researcher_table["use_for_orcid_sensitivity"] = (
    researcher_table["use_for_main_career_age"]
    & researcher_table["orcid_org_comparison"].eq("orcid_agrees")
)
researcher_table["use_for_ideal_sensitivity"] = (
    researcher_table["use_for_main_career_age"]
    & researcher_table["ideal_openalex_case"]
)

subgroup_summary = pd.DataFrame(
    [
        {
            "group": "all_pc_researchers",
            "n_researchers": len(researcher_table),
            "share_of_all_researchers": 1.0,
        },
        {
            "group": "main_career_age_subgroup",
            "n_researchers": int(researcher_table["use_for_main_career_age"].sum()),
            "share_of_all_researchers": float(researcher_table["use_for_main_career_age"].mean()),
        },
        {
            "group": "orcid_agreement_sensitivity",
            "n_researchers": int(researcher_table["use_for_orcid_sensitivity"].sum()),
            "share_of_all_researchers": float(researcher_table["use_for_orcid_sensitivity"].mean()),
        },
        {
            "group": "ideal_one_name_one_id_sensitivity",
            "n_researchers": int(researcher_table["use_for_ideal_sensitivity"].sum()),
            "share_of_all_researchers": float(researcher_table["use_for_ideal_sensitivity"].mean()),
        },
    ]
)

reliable_author_list = (
    researcher_table.loc[researcher_table["use_for_main_career_age"]]
    .sort_values(["name", "researcher_id"])
    .reset_index(drop=True)
)

output_cols = [
    "researcher_id",
    "name",
    "mapped_name_for_matching",
    "openalex_name",
    "openalex_id",
    "orcid_from_openalex",
    "orcid_from_orcid_org",
    "match_method",
    "match_confidence",
    "orcid_org_comparison",
    "ideal_openalex_case",
    "first_pc_year_any",
    "first_observed_pc_year",
    "n_pc_service_cells",
    "n_conferences_served",
    "use_for_orcid_sensitivity",
    "use_for_ideal_sensitivity",
]

def write_csv(frame, path, required_columns=None):
    schema_is_stale = False
    if path.exists() and required_columns is not None:
        existing_columns = set(pd.read_csv(path, nrows=0).columns)
        schema_is_stale = not set(required_columns).issubset(existing_columns)

    if path.exists() and not setup.overwrite_artifacts and not schema_is_stale:
        print(f"kept existing {path.relative_to(PROJECT)}")
        return
    if path.exists() and schema_is_stale:
        print(f"refreshing stale schema in {path.relative_to(PROJECT)}")
    frame.to_csv(path, index=False)
    print(f"wrote {path.relative_to(PROJECT)}")


def parquet_has_columns(path, required_columns):
    if not path.exists():
        return False
    existing_columns = set(pd.read_parquet(path).columns)
    return set(required_columns).issubset(existing_columns)

write_csv(reliable_author_list[output_cols], RELIABLE_AUTHOR_LIST_OUT)
write_csv(subgroup_summary, SUBGROUP_SUMMARY_OUT)

display(subgroup_summary)
display(reliable_author_list[output_cols].head())

wrote step_4_artifacts/summary_tables/career_age_reliable_author_list.csv
wrote step_4_artifacts/summary_tables/career_age_subgroup_summary.csv


,group,n_researchers,share_of_all_researchers
0,all_pc_researchers,952,1.000000
1,main_career_age_subgroup,790,0.829832
2,orcid_agreement_sensitivity,442,0.464286
3,ideal_one_name_one_id_sensitivity,448,0.470588


,researcher_id,name,mapped_name_for_matching,openalex_name,openalex_id,orcid_from_openalex,orcid_from_orcid_org,match_method,match_confidence,orcid_org_comparison,ideal_openalex_case,first_pc_year_any,first_observed_pc_year,n_pc_service_cells,n_conferences_served,use_for_orcid_sensitivity,use_for_ideal_sensitivity
0,aaronstump,Aaron Stump,Aaron Stump,Aaron Stump,A5072489480,0000-0002-9720-0003,0000-0002-9720-0003,exact_name_one_openalex_author_id,GREEN,orcid_agrees,True,2019.0,2019,2,1,True,True
1,abhinavverma1,Abhinav Verma,Abhinav Verma,Abhinav Verma,A5101988843,0000-0002-9820-8285,NaN,exact_name_one_openalex_author_id,YELLOW,orcid_org_not_unique,True,2023.0,2023,1,1,False,True
2,adamchlipala,Adam Chlipala,Adam Chlipala,Adam Chlipala,A5078100439,0000-0001-7085-9417,NaN,exact_name_one_openalex_author_id,YELLOW,orcid_org_not_unique,False,2017.0,2017,7,3,False,False
3,adamwelc,Adam Welc,Adam Welc,Adam Welc,A5078668785,0009-0005-0515-4994,0009-0005-0515-4994,exact_name_one_openalex_author_id,GREEN,orcid_agrees,True,2020.0,2020,2,1,True,True
4,adriansampson,Adrian Sampson,Adrian Sampson,Adrian Sampson,A5004782337,0000-0003-0837-8924,0000-0003-0837-8924,exact_name_one_openalex_author_id,GREEN,orcid_agrees,True,2018.0,2018,3,2,True,True


## 5. Fetch or read OpenAlex author metrics

This section measures career age as:

`first same-conference PC year - first OpenAlex Computer Science publication year`.

I also keep the raw earliest OpenAlex year as a check, because
raw author histories can include implausible old works from OpenAlex
author merge noise.

I also fetch current OpenAlex h-index as a dated cross check. I do not
use h-index as the main seniority variable because it is measured at
fetch time, after many of the PC service years in the panel.

In safe mode, the notebook does not fetch from OpenAlex. It writes the
target author list above and then uses a cached author metrics table
only if one already exists.

In [5]:
OPENALEX_API_KEY = os.environ.get("OPENALEX_API_KEY")
H_INDEX_FETCH_DATE = date.today().isoformat()
MIN_CAREER_AGE_PUBLICATION_YEAR = 1980

def short_author_id(openalex_id):
    if pd.isna(openalex_id):
        return None
    text = str(openalex_id).strip()
    return text.rstrip("/").split("/")[-1]


def author_url(openalex_id):
    short_id = short_author_id(openalex_id)
    if short_id is None:
        return None
    return f"https://openalex.org/{short_id}"


def first_work_cache_path_for_author(openalex_id):
    short_id = short_author_id(openalex_id)
    return STEP_4_FIRST_WORK_RAW / f"{short_id}.json"


def first_cs_work_cache_path_for_author(openalex_id):
    short_id = short_author_id(openalex_id)
    return STEP_4_FIRST_CS_WORK_RAW / f"{short_id}.json"


def profile_cache_path_for_author(openalex_id):
    short_id = short_author_id(openalex_id)
    return STEP_4_AUTHOR_PROFILE_RAW / f"{short_id}.json"


def parse_first_work_response(data):
    results = data.get("results", []) if isinstance(data, dict) else []
    if not results:
        return {
            "raw_first_publication_fetch_status": "no_works",
            "raw_first_publication_year": np.nan,
            "raw_first_publication_work_id": None,
        }

    work = results[0]
    return {
        "raw_first_publication_fetch_status": "ok",
        "raw_first_publication_year": work.get("publication_year"),
        "raw_first_publication_work_id": work.get("id"),
    }


def fetch_first_work(openalex_id, overwrite=False):
    import requests

    path = first_work_cache_path_for_author(openalex_id)
    if path.exists() and not overwrite:
        data = json.loads(path.read_text())
        parsed = parse_first_work_response(data)
        parsed["raw_first_publication_fetch_status"] = (
            "cached_" + parsed["raw_first_publication_fetch_status"]
        )
        return parsed

    params = {
        "filter": f"authorships.author.id:{author_url(openalex_id)}",
        "sort": "publication_year:asc",
        "per-page": 1,
    }
    if OPENALEX_API_KEY:
        params["api_key"] = OPENALEX_API_KEY

    response = requests.get(
        "https://api.openalex.org/works",
        params=params,
        timeout=30,
    )
    if response.status_code != 200:
        return {
            "raw_first_publication_fetch_status": f"http_{response.status_code}",
            "raw_first_publication_year": np.nan,
            "raw_first_publication_work_id": None,
        }

    data = response.json()
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False))
    parsed = parse_first_work_response(data)
    parsed["raw_first_publication_fetch_status"] = (
        "fetched_" + parsed["raw_first_publication_fetch_status"]
    )
    return parsed


def parse_first_cs_work_response(data):
    results = data.get("results", []) if isinstance(data, dict) else []
    if not results:
        return {
            "first_cs_publication_fetch_status": "no_works",
            "first_cs_publication_year": np.nan,
            "first_cs_publication_work_id": None,
        }

    work = results[0]
    return {
        "first_cs_publication_fetch_status": "ok",
        "first_cs_publication_year": work.get("publication_year"),
        "first_cs_publication_work_id": work.get("id"),
    }


def fetch_first_cs_work(openalex_id, overwrite=False):
    import requests

    path = first_cs_work_cache_path_for_author(openalex_id)
    if path.exists() and not overwrite:
        data = json.loads(path.read_text())
        parsed = parse_first_cs_work_response(data)
        parsed["first_cs_publication_fetch_status"] = (
            "cached_" + parsed["first_cs_publication_fetch_status"]
        )
        return parsed

    params = {
        "filter": (
            f"authorships.author.id:{author_url(openalex_id)},"
            "concepts.id:C41008148"
        ),
        "sort": "publication_year:asc",
        "per-page": 1,
    }
    if OPENALEX_API_KEY:
        params["api_key"] = OPENALEX_API_KEY

    response = requests.get(
        "https://api.openalex.org/works",
        params=params,
        timeout=30,
    )
    if response.status_code != 200:
        return {
            "first_cs_publication_fetch_status": f"http_{response.status_code}",
            "first_cs_publication_year": np.nan,
            "first_cs_publication_work_id": None,
        }

    data = response.json()
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False))
    parsed = parse_first_cs_work_response(data)
    parsed["first_cs_publication_fetch_status"] = (
        "fetched_" + parsed["first_cs_publication_fetch_status"]
    )
    return parsed


def parse_author_profile_response(payload):
    if not isinstance(payload, dict):
        return {
            "author_profile_fetch_status": "invalid_json",
            "h_index": np.nan,
            "h_index_fetch_date": None,
        }

    fetched_at = payload.get("fetched_at")
    data = payload.get("response", payload)
    stats = data.get("summary_stats", {}) if isinstance(data, dict) else {}

    return {
        "author_profile_fetch_status": "ok",
        "h_index": stats.get("h_index"),
        "h_index_fetch_date": fetched_at,
    }


def fetch_author_profile(openalex_id, overwrite=False):
    import requests

    path = profile_cache_path_for_author(openalex_id)
    if path.exists() and not overwrite:
        payload = json.loads(path.read_text())
        parsed = parse_author_profile_response(payload)
        parsed["author_profile_fetch_status"] = (
            "cached_" + parsed["author_profile_fetch_status"]
        )
        return parsed

    params = {}
    if OPENALEX_API_KEY:
        params["api_key"] = OPENALEX_API_KEY

    response = requests.get(
        f"https://api.openalex.org/authors/{short_author_id(openalex_id)}",
        params=params,
        timeout=30,
    )
    if response.status_code != 200:
        return {
            "author_profile_fetch_status": f"http_{response.status_code}",
            "h_index": np.nan,
            "h_index_fetch_date": None,
        }

    payload = {
        "fetched_at": H_INDEX_FETCH_DATE,
        "source": "OpenAlex Authors API",
        "response": response.json(),
    }
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False))
    parsed = parse_author_profile_response(payload)
    parsed["author_profile_fetch_status"] = (
        "fetched_" + parsed["author_profile_fetch_status"]
    )
    return parsed


REQUIRED_AUTHOR_METRIC_COLUMNS = {
    "researcher_id",
    "openalex_id",
    "raw_first_publication_year",
    "first_cs_publication_year",
    "first_cs_publication_work_id",
    "h_index",
    "h_index_fetch_date",
}
DERIVED_AUTHOR_METRIC_COLUMNS = {
    "career_age_publication_year",
    "career_age_publication_year_status",
}
force_author_metrics_rebuild = False
force_derived_metric_update = False


def write_parquet(frame, path, force=False):
    if path.exists() and not (setup.overwrite_data or force):
        print(f"kept existing {path.relative_to(PROJECT)}")
        return
    frame.to_parquet(path, index=False)
    print(f"wrote {path.relative_to(PROJECT)}")


if AUTHOR_METRICS_OUT.exists() and not setup.overwrite_data:
    author_metrics = pd.read_parquet(AUTHOR_METRICS_OUT)
    missing_metric_columns = REQUIRED_AUTHOR_METRIC_COLUMNS - set(author_metrics.columns)
    if missing_metric_columns:
        print(
            "Existing author-metrics table is missing columns: "
            f"{sorted(missing_metric_columns)}"
        )
        force_author_metrics_rebuild = True
        author_metrics = pd.DataFrame()
    else:
        missing_derived_columns = (
            DERIVED_AUTHOR_METRIC_COLUMNS - set(author_metrics.columns)
        )
        if missing_derived_columns:
            print(
                "Existing author-metrics table is missing derived columns: "
                f"{sorted(missing_derived_columns)}"
            )
            force_derived_metric_update = True
        print(f"loaded existing {AUTHOR_METRICS_OUT.relative_to(PROJECT)}")
elif not setup.allow_network:
    author_metrics = pd.DataFrame()
    fetch_summary = reliable_author_list[output_cols].copy()
    fetch_summary["raw_first_publication_fetch_status"] = (
        "not_fetched_network_disabled"
    )
    fetch_summary["first_cs_publication_fetch_status"] = (
        "not_fetched_network_disabled"
    )
    fetch_summary["author_profile_fetch_status"] = (
        "not_fetched_network_disabled"
    )
    fetch_summary["h_index"] = np.nan
    fetch_summary["h_index_fetch_date"] = None
    write_csv(fetch_summary, FETCH_SUMMARY_OUT)
    print(
        "Network is disabled and no author-metrics table was rebuilt. "
        "Set inputs.allow_network: true to fetch OpenAlex author metadata."
    )

if author_metrics.empty and setup.allow_network:
    targets = reliable_author_list.copy()
    if setup.openalex_sample_limit is not None:
        targets = targets.head(int(setup.openalex_sample_limit)).copy()
        print(f"Using OpenAlex sample limit: {len(targets)} authors")

    rows = []
    for index, row in targets.reset_index(drop=True).iterrows():
        first_work = fetch_first_work(
            row["openalex_id"],
            overwrite=setup.overwrite_data,
        )
        first_cs_work = fetch_first_cs_work(
            row["openalex_id"],
            overwrite=setup.overwrite_data,
        )
        profile = fetch_author_profile(
            row["openalex_id"],
            overwrite=setup.overwrite_data,
        )
        rows.append(
            {
                "researcher_id": row["researcher_id"],
                "name": row["name"],
                "openalex_id": row["openalex_id"],
                "match_confidence": row["match_confidence"],
                "orcid_org_comparison": row["orcid_org_comparison"],
                "ideal_openalex_case": row["ideal_openalex_case"],
                **first_work,
                **first_cs_work,
                **profile,
            }
        )
        if (index + 1) % 25 == 0 or index + 1 == len(targets):
            print(f"processed {index + 1:,} / {len(targets):,} authors")
        time.sleep(0.05)

    author_metrics = pd.DataFrame(rows)
    write_parquet(
        author_metrics,
        AUTHOR_METRICS_OUT,
        force=force_author_metrics_rebuild,
    )
    write_csv(author_metrics, FETCH_SUMMARY_OUT)

if not author_metrics.empty:
    author_metrics["raw_first_publication_year"] = pd.to_numeric(
        author_metrics["raw_first_publication_year"], errors="coerce"
    )
    author_metrics["first_cs_publication_year"] = pd.to_numeric(
        author_metrics["first_cs_publication_year"], errors="coerce"
    )
    author_metrics["h_index"] = pd.to_numeric(
        author_metrics["h_index"], errors="coerce"
    )
    author_metrics["career_age_publication_year"] = author_metrics[
        "first_cs_publication_year"
    ].where(
        author_metrics["first_cs_publication_year"].ge(
            MIN_CAREER_AGE_PUBLICATION_YEAR
        )
    )
    author_metrics["career_age_publication_year_status"] = np.select(
        [
            author_metrics["first_cs_publication_year"].isna(),
            author_metrics["first_cs_publication_year"].lt(
                MIN_CAREER_AGE_PUBLICATION_YEAR
            ),
        ],
        [
            "missing_first_cs_publication_year",
            "excluded_pre_1980_openalex_noise",
        ],
        default="usable_1980_or_later",
    )
    if force_derived_metric_update:
        write_parquet(author_metrics, AUTHOR_METRICS_OUT, force=True)
    display(author_metrics.head())
    display(
        author_metrics[
            [
                "raw_first_publication_fetch_status",
                "first_cs_publication_fetch_status",
                "author_profile_fetch_status",
            ]
        ]
        .value_counts(dropna=False)
        .reset_index(name="n_researchers")
    )

loaded existing step_4_data/prepared/researcher_openalex_author_metrics.parquet


,researcher_id,name,openalex_id,match_confidence,orcid_org_comparison,ideal_openalex_case,raw_first_publication_fetch_status,raw_first_publication_year,raw_first_publication_work_id,first_cs_publication_fetch_status,first_cs_publication_year,first_cs_publication_work_id,author_profile_fetch_status,h_index,h_index_fetch_date,career_age_publication_year,career_age_publication_year_status
0,aaronstump,Aaron Stump,A5072489480,GREEN,orcid_agrees,True,cached_ok,1999,https://openalex.org/W1631146593,fetched_ok,2000,https://openalex.org/W1534557881,cached_ok,25,2026-06-12,2000.0,usable_1980_or_later
1,abhinavverma1,Abhinav Verma,A5101988843,YELLOW,orcid_org_not_unique,True,cached_ok,2008,https://openalex.org/W2151091462,fetched_ok,2011,https://openalex.org/W1901947321,cached_ok,9,2026-06-12,2011.0,usable_1980_or_later
2,adamchlipala,Adam Chlipala,A5078100439,YELLOW,orcid_org_not_unique,False,cached_ok,2004,https://openalex.org/W1556262481,fetched_ok,2004,https://openalex.org/W1556262481,cached_ok,29,2026-06-12,2004.0,usable_1980_or_later
3,adamwelc,Adam Welc,A5078668785,GREEN,orcid_agrees,True,cached_ok,2004,https://openalex.org/W4245339677,fetched_ok,2004,https://openalex.org/W4245339677,cached_ok,20,2026-06-12,2004.0,usable_1980_or_later
4,adriansampson,Adrian Sampson,A5004782337,GREEN,orcid_agrees,True,cached_ok,2008,https://openalex.org/W2110880714,fetched_ok,2008,https://openalex.org/W2110880714,cached_ok,22,2026-06-12,2008.0,usable_1980_or_later


,raw_first_publication_fetch_status,first_cs_publication_fetch_status,author_profile_fetch_status,n_researchers
0,cached_ok,fetched_ok,cached_ok,790


## 6. Build the balanced event study rows

The event unit is a researcher and conference pair. I align each pair on
the first PC service year in that same conference. The event study
sample keeps only pairs observed at all five event times:

`t = -2, -1, 0, +1, +2`.

I also exclude event units where the researcher serves on the same
conference PC again at `t = +1` or `t = +2`. This keeps the period after PC
closer to an isolated first observed PC event, rather than mixing
first service with continued or repeated PC service.

In [6]:
EVENT_TIMES = np.array([-2, -1, 0, 1, 2], dtype=int)
BASELINE_TIMES = {-2, -1}
FOLLOWUP_TIMES = {1, 2}
EVENT_DEFINITION = "balanced_no_followup_pc_t1_t2"
BOOTSTRAP_REPS = 2000


def bootstrap_summary(matrix, event_times, reps=BOOTSTRAP_REPS, seed=1234):
    if matrix.size == 0:
        return pd.DataFrame()

    rng = np.random.default_rng(seed)
    n_units = matrix.shape[0]
    boot = np.empty((reps, matrix.shape[1]))
    for draw in range(reps):
        idx = rng.integers(0, n_units, size=n_units)
        boot[draw] = np.nanmean(matrix[idx], axis=0)

    return pd.DataFrame(
        {
            "event_time": event_times,
            "mean": np.nanmean(matrix, axis=0),
            "ci_lower": np.nanpercentile(boot, 2.5, axis=0),
            "ci_upper": np.nanpercentile(boot, 97.5, axis=0),
            "n_event_units": n_units,
        }
    )


if author_metrics.empty:
    balanced_event_rows = pd.DataFrame()
    event_unit_summary = pd.DataFrame()
    log_summary = pd.DataFrame()
    raw_summary = pd.DataFrame()
    log_by_conference = pd.DataFrame()
    raw_by_conference = pd.DataFrame()
    print("Skipping event-study construction until author metrics are available.")
else:
    author_metrics_keep = author_metrics[
        [
            "researcher_id",
            "raw_first_publication_year",
            "raw_first_publication_work_id",
            "first_cs_publication_year",
            "first_cs_publication_work_id",
            "career_age_publication_year",
            "career_age_publication_year_status",
            "h_index",
            "h_index_fetch_date",
            "orcid_org_comparison",
            "ideal_openalex_case",
            "raw_first_publication_fetch_status",
            "first_cs_publication_fetch_status",
            "author_profile_fetch_status",
        ]
    ].copy()

    event_panel = panel_full.merge(author_metrics_keep, on="researcher_id", how="inner")
    event_panel = event_panel.loc[
        event_panel["researcher_id"].isin(reliable_author_list["researcher_id"])
    ].copy()
    event_panel = event_panel.loc[
        event_panel["first_pc_year_conference"].notna()
        & event_panel["citation_count"].notna()
        & event_panel["career_age_publication_year"].notna()
    ].copy()
    event_panel["first_pc_year_conference"] = (
        event_panel["first_pc_year_conference"].astype(int)
    )
    event_panel["raw_first_publication_year"] = (
        event_panel["raw_first_publication_year"].astype("Int64")
    )
    event_panel["first_cs_publication_year"] = (
        event_panel["first_cs_publication_year"].astype(int)
    )
    event_panel["career_age_publication_year"] = (
        event_panel["career_age_publication_year"].astype(int)
    )
    event_panel["event_time"] = (
        event_panel["year"] - event_panel["first_pc_year_conference"]
    ).astype(int)
    event_panel = event_panel.loc[event_panel["event_time"].isin(EVENT_TIMES)].copy()
    event_panel["career_age_at_first_pc"] = (
        event_panel["first_pc_year_conference"]
        - event_panel["career_age_publication_year"]
    )
    event_panel = event_panel.loc[event_panel["career_age_at_first_pc"].ge(0)].copy()

    pc_year_counts = (
        panel.loc[panel["pc_status"].eq(1)]
        .groupby(["researcher_id", "conference"], as_index=False)
        .agg(n_pc_years_this_conference=("year", "nunique"))
    )
    event_panel = event_panel.merge(
        pc_year_counts,
        on=["researcher_id", "conference"],
        how="left",
        validate="many_to_one",
    )
    event_panel["n_pc_years_this_conference"] = (
        event_panel["n_pc_years_this_conference"].fillna(0).astype(int)
    )
    event_panel["service_pattern_this_conference"] = np.where(
        event_panel["n_pc_years_this_conference"].eq(1),
        "single_observed_pc_year",
        "repeat_observed_pc_years",
    )
    def clean_bool(series):
        return (
            series.astype("string")
            .str.lower()
            .map({"true": True, "false": False})
            .fillna(False)
            .astype(bool)
        )


    prior_pc_before_observed = clean_bool(
        event_panel["had_prior_pc_before_observed_conf"]
    )
    true_first_pc_in_observed_year = clean_bool(
        event_panel["is_true_first_pc_in_observed_year_conf"]
    )
    event_panel["source_history_pc_group_this_conference"] = np.select(
        [
            prior_pc_before_observed,
            true_first_pc_in_observed_year,
        ],
        [
            "prior_pc_found_before_observed_year",
            "source_verified_first_pc_year",
        ],
        default="no_prior_pc_evidence_found",
    )

    event_panel["event_unit_id"] = (
        event_panel["researcher_id"]
        + "|"
        + event_panel["conference"]
        + "|"
        + event_panel["first_pc_year_conference"].astype(str)
    )

    observed_times = (
        event_panel.groupby("event_unit_id")["event_time"]
        .agg(lambda values: set(values.astype(int)))
        .reset_index(name="observed_event_times")
    )
    observed_times["is_balanced_window"] = observed_times["observed_event_times"].map(
        lambda values: set(EVENT_TIMES).issubset(values)
    )
    followup_pc = (
        event_panel.loc[event_panel["event_time"].isin(FOLLOWUP_TIMES)]
        .groupby("event_unit_id")["pc_status"]
        .max()
        .reset_index(name="has_followup_pc_service")
    )
    observed_times = observed_times.merge(
        followup_pc,
        on="event_unit_id",
        how="left",
        validate="one_to_one",
    )
    observed_times["has_followup_pc_service"] = (
        observed_times["has_followup_pc_service"].fillna(0).astype(int)
    )
    observed_times["is_isolated_first_pc_event_window"] = (
        observed_times["is_balanced_window"]
        & observed_times["has_followup_pc_service"].eq(0)
    )
    balanced_unit_ids = observed_times.loc[
        observed_times["is_isolated_first_pc_event_window"], "event_unit_id"
    ]

    balanced_event_rows = event_panel.loc[
        event_panel["event_unit_id"].isin(balanced_unit_ids)
    ].copy()
    event_window_flags = observed_times[
        [
            "event_unit_id",
            "is_balanced_window",
            "has_followup_pc_service",
            "is_isolated_first_pc_event_window",
        ]
    ].copy()
    balanced_event_rows = balanced_event_rows.merge(
        event_window_flags,
        on="event_unit_id",
        how="left",
        validate="many_to_one",
    )
    balanced_event_rows["log10_citations"] = np.log10(
        balanced_event_rows["citation_count"] + 1
    )

    baselines = (
        balanced_event_rows.loc[balanced_event_rows["event_time"].isin(BASELINE_TIMES)]
        .groupby("event_unit_id", as_index=False)
        .agg(
            baseline_raw_citations=("citation_count", "mean"),
            baseline_log10_citations=("log10_citations", "mean"),
        )
    )
    balanced_event_rows = balanced_event_rows.merge(
        baselines, on="event_unit_id", how="left", validate="many_to_one"
    )
    balanced_event_rows["delta_raw_citations"] = (
        balanced_event_rows["citation_count"]
        - balanced_event_rows["baseline_raw_citations"]
    )
    balanced_event_rows["delta_log10_citations"] = (
        balanced_event_rows["log10_citations"]
        - balanced_event_rows["baseline_log10_citations"]
    )
    balanced_event_rows["event_definition"] = EVENT_DEFINITION

    event_unit_summary = (
        balanced_event_rows.drop_duplicates("event_unit_id")
        [
            [
                "event_unit_id",
                "researcher_id",
                "name",
                "conference",
                "first_pc_year_conference",
                "raw_first_publication_year",
                "first_cs_publication_year",
                "career_age_publication_year",
                "career_age_publication_year_status",
                "career_age_at_first_pc",
                "h_index",
                "h_index_fetch_date",
                "n_pc_years_this_conference",
                "service_pattern_this_conference",
                "first_observed_pc_year_conf",
                "source_verified_first_pc_year_conf",
                "source_verified_first_broad_service_year_conf",
                "had_prior_pc_before_observed_conf",
                "had_prior_broad_service_before_observed_conf",
                "prior_pc_evidence_count_conf",
                "prior_broad_service_evidence_count_conf",
                "is_true_first_pc_in_observed_year_conf",
                "is_true_first_broad_service_in_observed_year_conf",
                "source_history_pc_group_this_conference",
                "is_balanced_window",
                "has_followup_pc_service",
                "is_isolated_first_pc_event_window",
                "event_definition",
                "match_confidence",
                "orcid_org_comparison",
                "ideal_openalex_case",
                "baseline_raw_citations",
                "baseline_log10_citations",
            ]
        ]
        .sort_values(["conference", "first_pc_year_conference", "name"])
    )

    log_matrix = (
        balanced_event_rows.pivot_table(
            index="event_unit_id",
            columns="event_time",
            values="delta_log10_citations",
            aggfunc="mean",
        )
        .reindex(columns=EVENT_TIMES)
        .to_numpy()
    )
    raw_matrix = (
        balanced_event_rows.pivot_table(
            index="event_unit_id",
            columns="event_time",
            values="delta_raw_citations",
            aggfunc="mean",
        )
        .reindex(columns=EVENT_TIMES)
        .to_numpy()
    )

    log_summary = bootstrap_summary(log_matrix, EVENT_TIMES, seed=1234)
    raw_summary = bootstrap_summary(raw_matrix, EVENT_TIMES, seed=1234)
    for summary_frame in [log_summary, raw_summary]:
        if not summary_frame.empty:
            summary_frame.insert(0, "event_definition", EVENT_DEFINITION)

    def summarize_by_conference(rows, value_col, seed_base):
        pieces = []
        for idx, (conference, conference_rows) in enumerate(
            sorted(rows.groupby("conference"), key=lambda item: item[0])
        ):
            matrix = (
                conference_rows.pivot_table(
                    index="event_unit_id",
                    columns="event_time",
                    values=value_col,
                    aggfunc="mean",
                )
                .reindex(columns=EVENT_TIMES)
                .to_numpy()
            )
            summary = bootstrap_summary(matrix, EVENT_TIMES, seed=1234)
            if not summary.empty:
                summary.insert(0, "conference", conference)
            pieces.append(summary)

        if not pieces:
            return pd.DataFrame()

        return pd.concat(pieces, ignore_index=True)


    log_by_conference = summarize_by_conference(
        balanced_event_rows,
        "delta_log10_citations",
        seed_base=1234,
    )
    raw_by_conference = summarize_by_conference(
        balanced_event_rows,
        "delta_raw_citations",
        seed_base=1234,
    )
    for summary_frame in [log_by_conference, raw_by_conference]:
        if not summary_frame.empty:
            summary_frame.insert(0, "event_definition", EVENT_DEFINITION)

    force_event_outputs = bool(
        globals().get("force_author_metrics_rebuild", False)
        or globals().get("force_derived_metric_update", False)
    )
    event_rows_required_columns = [
        "n_pc_years_this_conference",
        "service_pattern_this_conference",
        "source_history_pc_group_this_conference",
        "is_true_first_broad_service_in_observed_year_conf",
        "is_balanced_window",
        "has_followup_pc_service",
        "is_isolated_first_pc_event_window",
        "event_definition",
        "delta_log10_citations",
        "delta_raw_citations",
    ]
    event_rows_schema_ok = parquet_has_columns(
        EVENT_ROWS_OUT,
        event_rows_required_columns,
    )
    if (
        not (setup.overwrite_data or force_event_outputs)
        and EVENT_ROWS_OUT.exists()
        and event_rows_schema_ok
    ):
        print(f"kept existing {EVENT_ROWS_OUT.relative_to(PROJECT)}")
    else:
        if EVENT_ROWS_OUT.exists() and not event_rows_schema_ok:
            print(f"refreshing stale schema in {EVENT_ROWS_OUT.relative_to(PROJECT)}")
        balanced_event_rows.to_parquet(EVENT_ROWS_OUT, index=False)
        print(f"wrote {EVENT_ROWS_OUT.relative_to(PROJECT)}")

    write_csv(
        event_unit_summary,
        EVENT_UNITS_OUT,
        required_columns=[
            "n_pc_years_this_conference",
            "service_pattern_this_conference",
            "source_history_pc_group_this_conference",
            "is_true_first_broad_service_in_observed_year_conf",
            "is_balanced_window",
            "has_followup_pc_service",
            "is_isolated_first_pc_event_window",
            "event_definition",
        ],
    )
    write_csv(log_summary, EVENT_LOG_SUMMARY_OUT, required_columns=["event_definition"])
    write_csv(raw_summary, EVENT_RAW_SUMMARY_OUT, required_columns=["event_definition"])
    write_csv(
        log_by_conference,
        EVENT_LOG_BY_CONFERENCE_OUT,
        required_columns=["event_definition"],
    )
    write_csv(
        raw_by_conference,
        EVENT_RAW_BY_CONFERENCE_OUT,
        required_columns=["event_definition"],
    )

    print(f"balanced event units: {event_unit_summary.shape[0]:,}")
    display(event_unit_summary["conference"].value_counts().reset_index())
    display(log_summary)
    display(raw_summary)
    display(log_by_conference.head())

kept existing step_4_data/prepared/career_age_event_window_rows.parquet
wrote step_4_artifacts/summary_tables/career_age_event_unit_summary.csv
wrote step_4_artifacts/summary_tables/career_age_event_study_log_summary.csv
wrote step_4_artifacts/summary_tables/career_age_event_study_raw_summary.csv
wrote step_4_artifacts/summary_tables/career_age_log_by_conf.csv
wrote step_4_artifacts/summary_tables/career_age_raw_by_conf.csv
balanced event units: 403


,conference,count
0,PLDI,138
1,POPL,138
2,ICFP,106
3,OOPSLA,21


,event_definition,event_time,mean,ci_lower,ci_upper,n_event_units
0,balanced_no_followup_pc_t1_t2,-2,-0.005161,-0.024241,0.013441,403
1,balanced_no_followup_pc_t1_t2,-1,0.005161,-0.013441,0.024241,403
2,balanced_no_followup_pc_t1_t2,0,0.031961,0.000523,0.063143,403
3,balanced_no_followup_pc_t1_t2,1,0.019118,-0.015601,0.053582,403
4,balanced_no_followup_pc_t1_t2,2,0.003049,-0.033274,0.039667,403


,event_definition,event_time,mean,ci_lower,ci_upper,n_event_units
0,balanced_no_followup_pc_t1_t2,-2,0.114144,-0.176179,0.410732,403
1,balanced_no_followup_pc_t1_t2,-1,-0.114144,-0.410732,0.176179,403
2,balanced_no_followup_pc_t1_t2,0,0.176179,-0.302761,0.616656,403
3,balanced_no_followup_pc_t1_t2,1,0.270471,-0.327543,0.858623,403
4,balanced_no_followup_pc_t1_t2,2,0.114144,-0.442990,0.658902,403


,event_definition,conference,event_time,mean,ci_lower,ci_upper,n_event_units
0,balanced_no_followup_pc_t1_t2,ICFP,-2,-0.016997,-0.053386,0.018000,106
1,balanced_no_followup_pc_t1_t2,ICFP,-1,0.016997,-0.018000,0.053386,106
2,balanced_no_followup_pc_t1_t2,ICFP,0,0.082709,0.023922,0.144153,106
3,balanced_no_followup_pc_t1_t2,ICFP,1,0.025469,-0.047923,0.092110,106
4,balanced_no_followup_pc_t1_t2,ICFP,2,0.030813,-0.034259,0.094456,106


## 7. Plot the conference specific career age event study analyses

In [7]:
def summarize_plot_rows(rows, value_col, seed):
    if rows.empty:
        return pd.DataFrame()

    matrix = (
        rows.pivot_table(
            index="event_unit_id",
            columns="event_time",
            values=value_col,
            aggfunc="mean",
        )
        .reindex(columns=EVENT_TIMES)
        .to_numpy()
    )
    return bootstrap_summary(matrix, EVENT_TIMES, seed=seed)


def draw_event_axis(
    ax,
    summary,
    color,
    title,
    ylabel=r"$\Delta \log_{10}(\mathrm{citations}+1)$",
):
    if summary.empty:
        ax.text(
            0.5,
            0.5,
            "No event units",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        ax.axis("off")
        return

    ax.axhline(0, color="black", lw=0.9)
    ax.axvline(0, color="black", lw=0.9, ls="--")
    ax.scatter(summary["event_time"], summary["mean"], color=color, s=28, zorder=3)
    post_summary = summary.loc[summary["event_time"].ge(0)].copy()
    if not post_summary.empty:
        yerr = np.vstack(
            [
                post_summary["mean"].to_numpy()
                - post_summary["ci_lower"].to_numpy(),
                post_summary["ci_upper"].to_numpy()
                - post_summary["mean"].to_numpy(),
            ]
        )
        ax.errorbar(
            post_summary["event_time"],
            post_summary["mean"],
            yerr=yerr,
            fmt="none",
            ecolor="black",
            elinewidth=0.9,
            capsize=3.5,
            capthick=0.9,
            zorder=4,
        )
    ax.set_xticks(EVENT_TIMES)
    ax.set_xlabel("Event time")
    ax.set_ylabel(ylabel)
    ax.set_title(title, loc="left")
    ax.grid(color="#DDDDDD", ls=":", lw=0.7)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def draw_hist_axis(ax, values, bins, color, title, xlabel, xlim=None):
    values = pd.Series(values).dropna()
    if values.empty:
        ax.text(
            0.5,
            0.5,
            "No data",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        ax.axis("off")
        return

    counts, _, _ = ax.hist(
        values,
        bins=bins,
        color=color,
        alpha=0.22,
        edgecolor="white",
        linewidth=0.4,
    )
    max_count = max(float(np.max(counts)), 1.0)
    rng = np.random.default_rng(1234)
    point_y = rng.uniform(0.015 * max_count, 0.09 * max_count, len(values))
    ax.scatter(
        values,
        point_y,
        s=8,
        color="#333333",
        alpha=0.35,
        linewidth=0,
        zorder=3,
    )
    ax.axvline(values.median(), color="black", lw=1.1, label="median")
    ax.axvline(values.mean(), color=color, lw=1.2, ls="--", label="mean")
    ax.set_title(
        f"{title}\nmedian={values.median():.1f}, mean={values.mean():.1f}",
        loc="left",
    )
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Event units")
    if xlim is not None:
        ax.set_xlim(xlim)
    ax.set_ylim(0, max_count * 1.18)
    ax.grid(color="#DDDDDD", ls=":", lw=0.7)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


if balanced_event_rows.empty or event_unit_summary.empty:
    print("Skipping plots until event-study rows are available.")
else:
    h_index_dates = (
        event_unit_summary["h_index_fetch_date"]
        .dropna()
        .astype(str)
        .sort_values()
        .unique()
    )
    h_index_date_label = h_index_dates[-1] if len(h_index_dates) else "unknown date"

    figure_records = []

    def assign_plot_group(frame):
        conference = frame["conference"].astype(str)
        first_year = pd.to_numeric(
            frame["first_pc_year_conference"],
            errors="coerce",
        )
        return np.select(
            [
                conference.eq("PLDI") & first_year.le(2020),
                conference.eq("PLDI") & first_year.ge(2021),
            ],
            [
                "PLDI before 2021",
                "PLDI 2021 onward",
            ],
            default=conference,
        )


    balanced_event_rows["plot_group"] = assign_plot_group(balanced_event_rows)
    event_unit_summary["plot_group"] = assign_plot_group(event_unit_summary)

    plot_group_slugs = {
        "POPL": "popl",
        "ICFP": "icfp",
        "OOPSLA": "oopsla",
        "PLDI before 2021": "pldi_before_2021",
        "PLDI 2021 onward": "pldi_2021_onward",
    }
    plot_group_order = [
        "POPL",
        "ICFP",
        "OOPSLA",
        "PLDI before 2021",
        "PLDI 2021 onward",
    ]
    available_plot_groups = [
        group
        for group in plot_group_order
        if group in set(event_unit_summary["plot_group"])
    ]
    max_career_age = event_unit_summary["career_age_at_first_pc"].max()
    career_age_upper = int(np.ceil(max_career_age / 5) * 5)
    career_age_upper = max(10, career_age_upper)
    career_age_bins = list(range(0, career_age_upper + 5, 5))
    max_h_index = event_unit_summary["h_index"].max()
    h_index_upper = int(np.ceil(max_h_index / 10) * 10)
    h_index_upper = max(10, h_index_upper)
    h_index_bins = list(range(0, h_index_upper + 10, 10))
    max_baseline = event_unit_summary["baseline_raw_citations"].max()
    baseline_upper = int(np.ceil(max_baseline / 5) * 5)
    baseline_upper = max(5, baseline_upper)
    baseline_bins = list(range(0, baseline_upper + 5, 5))

    def broad_first_service_mask(rows):
        broad_first = pd.to_numeric(
            rows["source_verified_first_broad_service_year_conf"],
            errors="coerce",
        )
        observed_first = pd.to_numeric(
            rows["first_observed_pc_year_conf"],
            errors="coerce",
        )
        return broad_first.eq(observed_first)


    sample_definitions = [
        {
            "sample_key": "all_balanced_event_units",
            "page_title": "All isolated balanced event units",
            "row_label": "All isolated balanced event units",
            "selector": lambda rows: rows,
            "seed_base": 1234,
        },
        {
            "sample_key": "no_earlier_main_pc_evidence",
            "page_title": "No earlier main-PC evidence found",
            "row_label": "No earlier main-PC evidence found",
            "selector": lambda rows: rows.loc[
                rows["source_history_pc_group_this_conference"].eq(
                    "source_verified_first_pc_year"
                )
            ],
            "seed_base": 1234,
        },
        {
            "sample_key": "no_earlier_broad_service_evidence",
            "page_title": "No earlier service",
            "row_label": "No earlier service",
            "selector": lambda rows: rows.loc[broad_first_service_mask(rows)],
            "seed_base": 1234,
        },
    ]


    def draw_sample_page(
        *,
        plot_group,
        plot_group_rows,
        sample_definition,
        group_idx,
        page_idx,
        color,
        split_kind,
    ):
        sample_rows = sample_definition["selector"](plot_group_rows).copy()
        sample_units = sample_rows.drop_duplicates("event_unit_id")
        if split_kind != "fixed_buckets":
            raise ValueError(f"Unknown split_kind: {split_kind}")
        median_career_age = sample_units["career_age_at_first_pc"].median()
        career_age = sample_rows["career_age_at_first_pc"]
        cohorts = [
            (
                "all_in_sample",
                sample_definition["row_label"],
                sample_rows,
            ),
            (
                "career_age_0_9",
                "Career age 0-9 years",
                sample_rows.loc[career_age.lt(10)],
            ),
            (
                "career_age_10_19",
                "Career age 10-19 years",
                sample_rows.loc[career_age.ge(10) & career_age.lt(20)],
            ),
            (
                "career_age_20_plus",
                "Career age 20+ years",
                sample_rows.loc[career_age.ge(20)],
            ),
        ]

        fig, axes = plt.subplots(
            nrows=len(cohorts),
            ncols=4,
            figsize=(15.2, 2.75 * len(cohorts) + 0.55),
            gridspec_kw={"width_ratios": [2.15, 1.0, 1.0, 1.05]},
        )
        axes = np.atleast_2d(axes)

        for row_idx, (cohort_key, cohort_title, cohort_rows) in enumerate(cohorts):
            cohort_units = cohort_rows.drop_duplicates("event_unit_id")
            n_units = int(cohort_units.shape[0])
            summary = summarize_plot_rows(
                cohort_rows,
                "delta_log10_citations",
                seed=1234,
            )

            draw_event_axis(
                axes[row_idx, 0],
                summary,
                color,
                f"{cohort_title}  |  N={n_units}",
            )
            draw_hist_axis(
                axes[row_idx, 1],
                cohort_units["career_age_at_first_pc"],
                bins=career_age_bins,
                color=color,
                title="Career age at first PC",
                xlabel="Years",
                xlim=(0, career_age_upper),
            )
            draw_hist_axis(
                axes[row_idx, 2],
                cohort_units["h_index"],
                bins=h_index_bins,
                color=color,
                title=f"h-index\nOpenAlex snapshot {h_index_date_label}",
                xlabel="h-index",
                xlim=(0, h_index_upper),
            )
            draw_hist_axis(
                axes[row_idx, 3],
                cohort_units["baseline_raw_citations"],
                bins=baseline_bins,
                color=color,
                title="Raw pre-PC baseline",
                xlabel="Mean citations at t=-2,-1",
                xlim=(0, baseline_upper),
            )

            figure_records.append(
                {
                    "plot_group": plot_group,
                    "sample": sample_definition["sample_key"],
                    "split_kind": split_kind,
                    "cohort": cohort_key,
                    "median_career_age": median_career_age,
                    "n_event_units": n_units,
                }
            )

        fig.suptitle(
            f"{plot_group}: {sample_definition['page_title']}",
            fontsize=13,
            y=0.99,
        )
        fig.tight_layout(rect=[0, 0, 1, 0.96], h_pad=2.1, w_pad=1.8)
        return fig


    figure_families = [
        {
            "split_kind": "fixed_buckets",
            "file_suffix": "fixed_buckets",
        },
    ]
    refresh_event_figures = True

    for family in figure_families:
        for group_idx, plot_group in enumerate(available_plot_groups):
            plot_group_rows = balanced_event_rows.loc[
                balanced_event_rows["plot_group"].eq(plot_group)
            ].copy()
            color = CONFERENCE_COLORS.get(plot_group, MAIN_COLOR)
            figure_out = (
                PRE_PC_MEAN_BASELINE_FIGURES
                / f"{plot_group_slugs[plot_group]}_{family['file_suffix']}.pdf"
            )

            if (
                figure_out.exists()
                and not setup.overwrite_artifacts
                and not refresh_event_figures
            ):
                print(f"kept existing {figure_out.relative_to(PROJECT)}")
                continue

            with PdfPages(figure_out) as pdf:
                for page_idx, sample_definition in enumerate(sample_definitions):
                    fig = draw_sample_page(
                        plot_group=plot_group,
                        plot_group_rows=plot_group_rows,
                        sample_definition=sample_definition,
                        group_idx=group_idx,
                        page_idx=page_idx,
                        color=color,
                        split_kind=family["split_kind"],
                    )
                    pdf.savefig(fig, bbox_inches="tight")
                    plt.close(fig)
            print(f"wrote {figure_out.relative_to(PROJECT)}")

    display(pd.DataFrame(figure_records))

wrote step_4_artifacts/figures_pre_pc_mean_baseline_event_study/popl_fixed_buckets.pdf
wrote step_4_artifacts/figures_pre_pc_mean_baseline_event_study/icfp_fixed_buckets.pdf
wrote step_4_artifacts/figures_pre_pc_mean_baseline_event_study/oopsla_fixed_buckets.pdf
wrote step_4_artifacts/figures_pre_pc_mean_baseline_event_study/pldi_before_2021_fixed_buckets.pdf
wrote step_4_artifacts/figures_pre_pc_mean_baseline_event_study/pldi_2021_onward_fixed_buckets.pdf


,plot_group,sample,split_kind,cohort,median_career_age,n_event_units
0,POPL,all_balanced_event_units,fixed_buckets,all_in_sample,17.0,138
1,POPL,all_balanced_event_units,fixed_buckets,career_age_0_9,17.0,18
2,POPL,all_balanced_event_units,fixed_buckets,career_age_10_19,17.0,64
3,POPL,all_balanced_event_units,fixed_buckets,career_age_20_plus,17.0,56
4,POPL,no_earlier_main_pc_evidence,fixed_buckets,all_in_sample,14.0,91
5,POPL,no_earlier_main_pc_evidence,fixed_buckets,career_age_0_9,14.0,18
6,POPL,no_earlier_main_pc_evidence,fixed_buckets,career_age_10_19,14.0,51
7,POPL,no_earlier_main_pc_evidence,fixed_buckets,career_age_20_plus,14.0,22
8,POPL,no_earlier_broad_service_evidence,fixed_buckets,all_in_sample,12.5,28
9,POPL,no_earlier_broad_service_evidence,fixed_buckets,career_age_0_9,12.5,7


## 8. Plot the main text ICFP slicing sensitivity check

In [8]:
def build_selected_large_jump_examples(panel_source, author_metric_source):
    selected_specs = [
        {
            "selection_rank": 1,
            "researcher_id": "ezgicicek",
            "name": "Ezgi Çiçek",
            "conference": "ICFP",
            "pc_year": 2019,
            "example_note": (
                "Balanced example from the no-earlier-service, "
                "0-9 year career-age slice."
            ),
        },
        {
            "selection_rank": 2,
            "researcher_id": "malgorzatabiernacka",
            "name": "Malgorzata Biernacka",
            "conference": "ICFP",
            "pc_year": 2022,
            "example_note": (
                "Illustrative raw-count example; earlier broad-service "
                "evidence is present."
            ),
        },
        {
            "selection_rank": 3,
            "researcher_id": "peterthiemann",
            "name": "Peter Thiemann",
            "conference": "ICFP",
            "pc_year": 2023,
            "example_note": (
                "Illustrative raw-count example added for the selected "
                "trajectory checks."
            ),
        },
        {
            "selection_rank": 4,
            "researcher_id": "martinelsman",
            "name": "Martin Elsman",
            "conference": "ICFP",
            "pc_year": 2018,
            "example_note": (
                "Illustrative first-PC example; later 2024 PC service is "
                "outside the +2 window, but t=-2 is unavailable because "
                "the panel starts in 2017."
            ),
        },
    ]

    records = []
    metric_cols = [
        "researcher_id",
        "raw_first_publication_year",
        "first_cs_publication_year",
        "career_age_publication_year",
        "career_age_publication_year_status",
        "h_index",
    ]
    metrics = author_metric_source[metric_cols].drop_duplicates("researcher_id").copy()
    metrics_by_researcher = metrics.set_index("researcher_id").to_dict("index")

    for spec in selected_specs:
        all_rows = panel_source.loc[
            panel_source["researcher_id"].eq(spec["researcher_id"])
            & panel_source["conference"].eq(spec["conference"])
        ].copy()
        window_rows = all_rows.loc[
            all_rows["year"].between(spec["pc_year"] - 2, spec["pc_year"] + 2)
        ].copy()
        window_rows["event_time"] = window_rows["year"].astype(int) - spec["pc_year"]

        citation_by_event_time = (
            window_rows.set_index("event_time")["citation_count"].to_dict()
        )
        pc_status_by_event_time = (
            window_rows.set_index("event_time")["pc_status"].to_dict()
        )
        citation_values = {
            event_time: citation_by_event_time.get(event_time, np.nan)
            for event_time in EVENT_TIMES
        }
        pre_values = [
            citation_values[event_time]
            for event_time in BASELINE_TIMES
            if pd.notna(citation_values[event_time])
        ]
        baseline = float(np.mean(pre_values)) if pre_values else np.nan
        pc_year_jump = (
            float(citation_values[0] - baseline)
            if pd.notna(citation_values[0]) and pd.notna(baseline)
            else np.nan
        )
        same_conference_pc_years = sorted(
            all_rows.loc[all_rows["pc_status"].eq(1), "year"]
            .astype(int)
            .unique()
            .tolist()
        )
        has_followup_pc_service = any(
            int(pc_status_by_event_time.get(event_time, 0)) == 1
            for event_time in FOLLOWUP_TIMES
        )

        if not window_rows.empty:
            name = window_rows["name"].dropna().iloc[0]
        else:
            name = spec["name"]

        metric_row = metrics_by_researcher.get(spec["researcher_id"], {})
        career_age_publication_year = metric_row.get(
            "career_age_publication_year", np.nan
        )
        career_age_at_pc = (
            spec["pc_year"] - int(career_age_publication_year)
            if pd.notna(career_age_publication_year)
            else np.nan
        )
        career_age_label = (
            f"career age {int(career_age_at_pc)}"
            if pd.notna(career_age_at_pc)
            else "career age n/a"
        )

        records.append(
            {
                "selection_rank": spec["selection_rank"],
                "researcher_id": spec["researcher_id"],
                "name": name,
                "conference": spec["conference"],
                "pc_year": spec["pc_year"],
                "citation_t_minus_2": citation_values[-2],
                "citation_t_minus_1": citation_values[-1],
                "citation_t_0": citation_values[0],
                "citation_t_plus_1": citation_values[1],
                "citation_t_plus_2": citation_values[2],
                "baseline_raw_citations": baseline,
                "pc_year_jump_raw": pc_year_jump,
                "career_age_at_pc_year": career_age_at_pc,
                "career_age_label": career_age_label,
                "career_age_publication_year": career_age_publication_year,
                "career_age_publication_year_status": metric_row.get(
                    "career_age_publication_year_status", pd.NA
                ),
                "h_index": metric_row.get("h_index", pd.NA),
                "has_full_pre_window": all(
                    pd.notna(citation_values[event_time])
                    for event_time in BASELINE_TIMES
                ),
                "has_full_event_window": all(
                    pd.notna(citation_values[event_time])
                    for event_time in EVENT_TIMES
                ),
                "has_followup_pc_service_t1_t2": has_followup_pc_service,
                "same_conference_pc_years": "; ".join(
                    str(year) for year in same_conference_pc_years
                ),
                "example_note": spec["example_note"],
            }
        )
    return pd.DataFrame(records)


def valid_source_conferences(selected_conference):
    if selected_conference == "OOPSLA_COMBINED":
        return {"OOPSLA1", "OOPSLA2"}
    return {selected_conference}


def build_selected_citation_source_table(
    selected_examples,
    match_selected_conference=False,
):
    citation_keys = pd.read_parquet(CITATION_KEYS_PATH)
    ref_authors = pd.read_parquet(REF_AUTHORS_PATH)
    papers_full = pd.read_parquet(PAPERS_PATH)
    papers = papers_full[
        ["work_id", "doi", "title", "conference", "conference_year", "issue"]
    ].copy()
    papers = papers.rename(
        columns={
            "work_id": "citing_work_id",
            "doi": "citing_doi",
            "title": "citing_title",
            "conference": "citing_conference",
            "conference_year": "citing_year",
            "issue": "citing_issue",
        }
    )
    paper_authors = {}
    paper_author_names = {}
    for _, row in papers_full.iterrows():
        paper_authors[row["work_id"]] = {
            author.get("author_id")
            for author in row["authorships"]
            if author.get("author_id")
        }
        paper_author_names[row["work_id"]] = "; ".join(
            author.get("author_name") or author.get("display_name") or ""
            for author in row["authorships"]
            if author.get("author_name") or author.get("display_name")
        )

    cited_work_cache_dirs = [
        PROJECT / "step_2_data" / "raw" / "openalex_cache" / "2026-06-11" / "works",
        PROJECT / "step_2_data" / "raw" / "openalex_cache" / "works",
    ]
    cited_work_cache = {}

    def load_cited_work(work_id):
        if work_id in cited_work_cache:
            return cited_work_cache[work_id]
        for cache_dir in cited_work_cache_dirs:
            path = cache_dir / f"{work_id}.json"
            if path.exists():
                cited_work_cache[work_id] = json.loads(path.read_text())
                return cited_work_cache[work_id]
        cited_work_cache[work_id] = {}
        return cited_work_cache[work_id]

    selected_ids = selected_examples["researcher_id"].unique().tolist()
    selected_keys = citation_keys.loc[
        citation_keys["researcher_id"].isin(selected_ids)
    ].copy()

    refs = ref_authors.copy()
    refs["ref_author_name_norm"] = refs["ref_author_name"].map(normalize_name)
    refs["event_id"] = (
        refs["work_id"].astype(str)
        + "|"
        + refs["referenced_work_id"].astype(str)
        + "|"
        + refs["ref_author_id"].fillna("").astype(str)
        + "|"
        + refs["ref_author_name_norm"].fillna("").astype(str)
    )

    id_keys = selected_keys.loc[
        selected_keys["citation_key_type"].eq("openalex_author_id")
    ].copy()
    name_keys = selected_keys.loc[
        ~selected_keys["citation_key_type"].eq("openalex_author_id")
    ].copy()

    id_events = refs.merge(
        id_keys,
        left_on="ref_author_id",
        right_on="citation_key_value",
        how="inner",
    )
    name_events = refs.merge(
        name_keys,
        left_on="ref_author_name_norm",
        right_on="citation_key_value",
        how="inner",
    )
    citation_events = pd.concat([id_events, name_events], ignore_index=True)
    citation_events = citation_events.drop_duplicates(
        ["researcher_id", "work_id", "referenced_work_id"]
    ).copy()

    citation_events = citation_events.rename(
        columns={
            "work_id": "citing_work_id",
            "issue": "conference",
            "conference_year": "year",
        }
    )
    citation_events = citation_events.merge(
        papers,
        on="citing_work_id",
        how="left",
        validate="many_to_one",
    )

    selected_windows = []
    for _, row in selected_examples.iterrows():
        for event_time in EVENT_TIMES:
            record = {
                "researcher_id": row["researcher_id"],
                "name": row["name"],
                "pc_year": int(row["pc_year"]),
                "event_time": int(event_time),
                "year": int(row["pc_year"] + event_time),
            }
            if "conference" in selected_examples.columns:
                record["selected_conference"] = row["conference"]
            selected_windows.append(record)
    selected_windows = pd.DataFrame(selected_windows)

    selected_sources = citation_events.merge(
        selected_windows,
        on=["researcher_id", "year"],
        how="inner",
    )
    if match_selected_conference:
        selected_sources = selected_sources.loc[
            selected_sources.apply(
                lambda row: row["conference"]
                in valid_source_conferences(row["selected_conference"]),
                axis=1,
            )
        ].copy()
    else:
        selected_sources = selected_sources.loc[
            selected_sources["conference"].eq("ICFP")
        ].copy()

    selected_sources["cited_work_title"] = selected_sources[
        "referenced_work_id"
    ].map(
        lambda work_id: (
            load_cited_work(work_id).get("title")
            or load_cited_work(work_id).get("display_name")
            or pd.NA
        )
    )
    selected_sources["cited_work_doi"] = selected_sources[
        "referenced_work_id"
    ].map(lambda work_id: load_cited_work(work_id).get("doi") or pd.NA)
    selected_sources["cited_work_age_years"] = (
        selected_sources["year"] - selected_sources["cited_publication_year"]
    )
    selected_sources["citing_authors"] = selected_sources["citing_work_id"].map(
        paper_author_names
    )

    def coauthor_overlap(row):
        citing_author_ids = paper_authors.get(row["citing_work_id"], set())
        cited_rows = ref_authors.loc[
            ref_authors["work_id"].eq(row["citing_work_id"])
            & ref_authors["referenced_work_id"].eq(row["referenced_work_id"])
        ].copy()
        cited_author_ids = set(cited_rows["ref_author_id"].dropna().astype(str))
        overlap_ids = sorted(set(citing_author_ids) & cited_author_ids)
        overlap_names = sorted(
            cited_rows.loc[
                cited_rows["ref_author_id"].astype(str).isin(overlap_ids),
                "ref_author_name",
            ]
            .dropna()
            .unique()
            .tolist()
        )
        return pd.Series(
            {
                "has_citing_author_on_cited_work": bool(overlap_ids),
                "citing_author_overlap_names": "; ".join(overlap_names),
            }
        )

    selected_sources = pd.concat(
        [
            selected_sources,
            selected_sources.apply(coauthor_overlap, axis=1),
        ],
        axis=1,
    )

    output_cols = [
        "researcher_id",
        "name",
        "pc_year",
        "event_time",
        "year",
        "conference",
        "citing_work_id",
        "citing_title",
        "citing_authors",
        "citing_doi",
        "referenced_work_id",
        "cited_work_title",
        "cited_work_doi",
        "ref_author_name",
        "ref_author_id",
        "ref_orcid",
        "cited_publication_year",
        "cited_work_age_years",
        "citation_key_type",
        "citation_key_value",
        "author_position",
        "is_self_citation",
        "has_citing_author_on_cited_work",
        "citing_author_overlap_names",
    ]
    if "selected_conference" in selected_sources.columns:
        output_cols.insert(3, "selected_conference")
    return (
        selected_sources[output_cols]
        .sort_values(
            [
                "name",
                "pc_year",
                "event_time",
                "citing_title",
                "referenced_work_id",
            ]
        )
        .reset_index(drop=True)
    )


def build_selected_prior_work_summary(selected_sources):
    group_cols = [
        "researcher_id",
        "name",
        "pc_year",
        "event_time",
        "year",
        "referenced_work_id",
        "cited_work_title",
        "cited_work_doi",
        "cited_publication_year",
        "cited_work_age_years",
    ]

    def join_unique(values):
        return " | ".join(sorted(set(values.dropna().astype(str))))

    summary = (
        selected_sources.groupby(group_cols, dropna=False)
        .agg(
            reference_edges=("citing_work_id", "size"),
            unique_citing_papers=("citing_work_id", "nunique"),
            citing_papers=("citing_title", join_unique),
            citing_authors=("citing_authors", join_unique),
            direct_self_edges=("is_self_citation", "sum"),
            coauthor_mediated_edges=(
                "has_citing_author_on_cited_work",
                "sum",
            ),
            coauthor_overlap_names=("citing_author_overlap_names", join_unique),
        )
        .reset_index()
        .sort_values(
            [
                "name",
                "pc_year",
                "event_time",
                "cited_publication_year",
                "cited_work_title",
                "referenced_work_id",
            ]
        )
    )
    return summary


def build_selected_prior_work_compact_table(selected_sources):
    display_table = selected_sources.copy()
    display_table["cited_work_title_key"] = (
        display_table["cited_work_title"].fillna("").str.lower().str.strip()
    )

    group_cols = [
        "researcher_id",
        "name",
        "pc_year",
        "event_time",
        "year",
        "cited_work_title_key",
        "cited_work_title",
    ]

    def join_unique(values):
        return " | ".join(sorted(set(values.dropna().astype(str))))

    compact = (
        display_table.groupby(group_cols, dropna=False)
        .agg(
            cited_publication_years=(
                "cited_publication_year",
                lambda x: ", ".join(
                    str(int(value))
                    for value in sorted(set(x.dropna().astype(int)))
                ),
            ),
            cited_work_age_years=(
                "cited_work_age_years",
                lambda x: ", ".join(
                    str(int(value))
                    for value in sorted(set(x.dropna().astype(int)))
                ),
            ),
            referenced_work_ids=("referenced_work_id", join_unique),
            cited_work_dois=("cited_work_doi", join_unique),
            unique_citing_papers=("citing_work_id", "nunique"),
            reference_edges=("citing_work_id", "size"),
            direct_self_edges=("is_self_citation", "sum"),
            coauthor_mediated_edges=(
                "has_citing_author_on_cited_work",
                "sum",
            ),
            citing_papers=("citing_title", join_unique),
            citing_authors=("citing_authors", join_unique),
            coauthor_overlap_names=(
                "citing_author_overlap_names",
                join_unique,
            ),
        )
        .reset_index()
        .drop(columns=["cited_work_title_key"])
        .sort_values(
            [
                "name",
                "pc_year",
                "event_time",
                "cited_publication_years",
                "cited_work_title",
            ]
        )
    )
    return compact


def build_selected_citing_paper_summary(selected_sources):
    group_cols = [
        "researcher_id",
        "name",
        "pc_year",
        "event_time",
        "year",
        "citing_work_id",
        "citing_title",
        "citing_authors",
        "citing_doi",
    ]

    def format_years(values):
        years = sorted(set(values.dropna().astype(int)))
        return ", ".join(str(year) for year in years)

    records = []
    for group_key, group in selected_sources.groupby(group_cols, dropna=False):
        row = dict(zip(group_cols, group_key))
        prior_summary = (
            group.groupby("cited_work_title", dropna=False)
            .agg(
                cited_publication_years=(
                    "cited_publication_year",
                    format_years,
                ),
                reference_edges=("referenced_work_id", "size"),
            )
            .reset_index()
            .sort_values(
                [
                    "cited_publication_years",
                    "cited_work_title",
                ]
            )
        )
        prior_parts = []
        for _, prior_row in prior_summary.iterrows():
            title = prior_row["cited_work_title"]
            years = prior_row["cited_publication_years"]
            edges = int(prior_row["reference_edges"])
            prior_parts.append(f"{title} ({years}; E={edges})")

        row.update(
            {
                "unique_cited_prior_titles": int(
                    prior_summary["cited_work_title"].nunique()
                ),
                "reference_edges": int(len(group)),
                "direct_self_edges": int(group["is_self_citation"].sum()),
                "coauthor_mediated_edges": int(
                    group["has_citing_author_on_cited_work"].sum()
                ),
                "cited_prior_works": " | ".join(prior_parts),
            }
        )
        records.append(row)

    return (
        pd.DataFrame(records)
        .sort_values(["name", "pc_year", "event_time", "citing_title"])
        .reset_index(drop=True)
    )


def collapse_title_key(value):
    if pd.isna(value):
        return ""
    text = (
        str(value)
        .lower()
        .replace("-", " ")
        .replace(":", " ")
        .replace(",", " ")
        .replace(".", " ")
    )
    return " ".join(text.split())


def build_selected_trajectory_t0_source_summary(selected_sources):
    t0 = selected_sources.loc[selected_sources["event_time"].eq(0)].copy()
    if t0.empty:
        return pd.DataFrame()
    t0["cited_work_title_key"] = t0["cited_work_title"].map(collapse_title_key)
    collapsed = t0.drop_duplicates(
        [
            "researcher_id",
            "selected_conference",
            "pc_year",
            "citing_work_id",
            "cited_work_title_key",
        ]
    ).copy()

    group_cols = [
        "researcher_id",
        "name",
        "selected_conference",
        "pc_year",
    ]
    raw_summary = (
        t0.groupby(group_cols, dropna=False)
        .agg(
            raw_edges=("referenced_work_id", "size"),
            citing_papers=("citing_work_id", "nunique"),
            raw_self_edges=("is_self_citation", "sum"),
            raw_coauthor_edges=("has_citing_author_on_cited_work", "sum"),
        )
        .reset_index()
    )
    collapsed_summary = (
        collapsed.groupby(group_cols, dropna=False)
        .agg(
            title_collapsed_edges=("cited_work_title_key", "size"),
            collapsed_self_edges=("is_self_citation", "sum"),
            collapsed_coauthor_edges=(
                "has_citing_author_on_cited_work",
                "sum",
            ),
        )
        .reset_index()
    )
    summary = raw_summary.merge(
        collapsed_summary,
        on=group_cols,
        how="left",
        validate="one_to_one",
    )
    summary["duplicate_version_edges"] = (
        summary["raw_edges"] - summary["title_collapsed_edges"]
    )
    return summary.rename(columns={"selected_conference": "conference"})


def build_main_text_event_study_figure(
    rows,
    value_col,
    value_scale,
    ylabel,
    seed_offset=1234,
    include_selected_examples=False,
    example_source=None,
):
    icfp_rows = rows.loc[rows["conference"].eq("ICFP")].copy()
    sample_specs = [
        {
            "sample": "all_isolated_balanced_event_units",
            "sample_label": "All isolated balanced events",
            "title_line_1": "All isolated balanced events",
            "title_line_2": "Full sample",
            "selector": lambda frame: frame,
            "seed": 1234,
        },
        {
            "sample": "all_isolated_balanced_career_age_20_plus",
            "sample_label": "All isolated balanced events, career age 20+ years",
            "title_line_1": "All isolated balanced events",
            "title_line_2": "Career age 20+ years",
            "selector": lambda frame: frame.loc[
                frame["career_age_at_first_pc"].ge(20)
            ],
            "seed": 1234,
        },
        {
            "sample": "no_earlier_broad_service_career_age_0_9",
            "sample_label": "No earlier service, career age 0-9 years",
            "title_line_1": "No earlier service",
            "title_line_2": "Career age 0-9 years",
            "selector": lambda frame: frame.loc[
                broad_first_service_mask(frame)
                & frame["career_age_at_first_pc"].lt(10)
            ],
            "seed": 1234,
        },
        {
            "sample": "no_earlier_broad_service_career_age_10_19",
            "sample_label": "No earlier service, career age 10-19 years",
            "title_line_1": "No earlier service",
            "title_line_2": "Career age 10-19 years",
            "selector": lambda frame: frame.loc[
                broad_first_service_mask(frame)
                & frame["career_age_at_first_pc"].ge(10)
                & frame["career_age_at_first_pc"].lt(20)
            ],
            "seed": 1234,
        },
    ]

    summary_pieces = []
    plot_rows_by_sample = {}
    for spec in sample_specs:
        sample_rows = spec["selector"](icfp_rows).copy()
        plot_rows_by_sample[spec["sample"]] = sample_rows
        summary = summarize_plot_rows(
            sample_rows,
            value_col,
            seed=1234,
        )
        if summary.empty:
            continue
        summary.insert(0, "conference", "ICFP")
        summary.insert(1, "sample", spec["sample"])
        summary.insert(2, "sample_label", spec["sample_label"])
        summary.insert(3, "value_scale", value_scale)
        summary_pieces.append(summary)

    if summary_pieces:
        summary_table = pd.concat(summary_pieces, ignore_index=True)
    else:
        summary_table = pd.DataFrame()

    selected_examples = pd.DataFrame()
    if include_selected_examples and example_source is not None:
        selected_examples = build_selected_large_jump_examples(
            example_source, author_metrics
        )

    if include_selected_examples:
        fig = plt.figure(figsize=(9.0, 8.35))
        gs = fig.add_gridspec(
            nrows=3,
            ncols=6,
            height_ratios=[1.0, 1.0, 0.86],
            hspace=0.82,
            wspace=0.45,
        )
        axes = np.array(
            [
                fig.add_subplot(gs[0, 0:3]),
                fig.add_subplot(gs[0, 3:6]),
                fig.add_subplot(gs[1, 0:3]),
                fig.add_subplot(gs[1, 3:6]),
            ]
        )
        example_axes = [
            fig.add_subplot(gs[2, 0:2]),
            fig.add_subplot(gs[2, 2:4]),
            fig.add_subplot(gs[2, 4:6]),
        ]
        for ax in axes[1:]:
            ax.sharex(axes[0])
            ax.sharey(axes[0])
    else:
        fig, axes = plt.subplots(
            nrows=2,
            ncols=2,
            figsize=(9.0, 6.0),
            sharex=True,
            sharey=True,
        )
        axes = axes.ravel()
        example_axes = []

    color = CONFERENCE_COLORS["ICFP"]
    for idx, (ax, spec) in enumerate(zip(axes, sample_specs), start=1):
        sample_summary = summary_table.loc[
            summary_table["sample"].eq(spec["sample"])
        ].copy()
        n_units = int(
            plot_rows_by_sample[spec["sample"]]["event_unit_id"].nunique()
        )
        title = (
            f"{spec['title_line_1']}\n"
            f"{spec['title_line_2']} (N={n_units})"
        )
        draw_event_axis(
            ax,
            sample_summary,
            color,
            f"({chr(96 + idx)}) {title}",
            ylabel=ylabel,
        )
        ax.set_xlabel("Event time")
        ax.title.set_fontsize(7.8)
        ax.xaxis.label.set_size(8.0)
        ax.yaxis.label.set_size(8.0)
        ax.tick_params(axis="both", labelsize=7.5)

    axes[0].set_xlabel("")
    axes[1].set_xlabel("")
    if include_selected_examples:
        axes[2].set_xlabel("")
        axes[3].set_xlabel("")
    axes[1].set_ylabel("")
    axes[3].set_ylabel("")

    if include_selected_examples and not selected_examples.empty:
        y_cols = [
            "citation_t_minus_2",
            "citation_t_minus_1",
            "citation_t_0",
            "citation_t_plus_1",
            "citation_t_plus_2",
        ]
        y_max = selected_examples[y_cols].max().max()
        y_upper = max(4, int(np.ceil((y_max + 1) / 2) * 2))
        for ax_idx, (ax, (_, row)) in enumerate(
            zip(example_axes, selected_examples.iterrows()), start=5
        ):
            y = row[y_cols].to_numpy(dtype=float)
            ax.axvline(0, color="black", lw=0.8, ls="--", alpha=0.75)
            ax.plot(
                EVENT_TIMES,
                y,
                color=color,
                lw=1.7,
                marker="o",
                markersize=4.0,
                markeredgecolor="black",
                markeredgewidth=0.35,
                zorder=3,
            )
            ax.set_title(
                (
                    f"({chr(96 + ax_idx)}) {row['name']}\n"
                    f"ICFP {int(row['pc_year'])}; {row['career_age_label']}"
                ),
                loc="left",
                fontsize=7.5,
            )
            ax.set_xlim(EVENT_TIMES.min() - 0.25, EVENT_TIMES.max() + 0.25)
            ax.set_ylim(-0.35, y_upper)
            ax.set_xticks(EVENT_TIMES)
            ax.set_xlabel("Event time")
            if ax_idx == 5:
                ax.set_ylabel("Citations")
            else:
                ax.set_ylabel("")
            ax.xaxis.label.set_size(8.0)
            ax.yaxis.label.set_size(8.0)
            ax.tick_params(axis="both", labelsize=7.5)
            ax.grid(color="#DDDDDD", ls=":", lw=0.7)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

    handles = [
        plt.Line2D([0], [0], color="black", lw=0.9, ls="--"),
        plt.Line2D(
            [0],
            [0],
            color="black",
            lw=0.9,
            marker="_",
            markersize=8,
            linestyle="none",
        ),
    ]
    labels = ["first PC year", r"95\% bootstrap CI"]
    axes[0].legend(
        handles,
        labels,
        loc="lower left",
        ncol=1,
        frameon=False,
        fontsize=7.2,
        handlelength=1.6,
        labelspacing=0.25,
        borderaxespad=0.35,
    )
    if include_selected_examples:
        fig.text(
            0.08,
            0.335,
            "Selected PC-member citation trajectories",
            ha="left",
            va="center",
            fontsize=8.7,
        )
        fig.tight_layout(rect=[0, 0, 1, 0.96])
    else:
        fig.tight_layout(rect=[0, 0, 1, 0.96], w_pad=1.8, h_pad=1.8)
    return summary_table, fig, selected_examples


if balanced_event_rows.empty:
    print("Skipping main-text event-study figure until event-study rows are available.")
else:
    required_columns = [
        "conference",
        "sample",
        "sample_label",
        "value_scale",
        "event_time",
        "mean",
        "ci_lower",
        "ci_upper",
        "n_event_units",
    ]
    main_event_outputs = [
        {
            "value_col": "delta_log10_citations",
            "value_scale": "log10_citations_plus_one",
            "ylabel": r"$\Delta \log_{10}(\mathrm{citations}+1)$",
            "seed_offset": 1234,
            "summary_out": MAIN_EVENT_LOG_SUMMARY_OUT,
            "figure_out": MAIN_EVENT_LOG_FIGURE_OUT,
            "report_figure_out": REPORT_MAIN_EVENT_LOG_FIGURE_OUT,
            "include_selected_examples": False,
        },
        {
            "value_col": "delta_raw_citations",
            "value_scale": "raw_citation_count",
            "ylabel": r"$\Delta$ citations",
            "seed_offset": 1234,
            "summary_out": MAIN_EVENT_RAW_SUMMARY_OUT,
            "figure_out": MAIN_EVENT_RAW_FIGURE_OUT,
            "report_figure_out": REPORT_MAIN_EVENT_RAW_FIGURE_OUT,
            "include_selected_examples": False,
        },
    ]
    main_event_summaries = []
    main_event_selected_examples = pd.DataFrame()
    main_event_selected_citation_sources = pd.DataFrame()
    main_event_selected_prior_work_summary = pd.DataFrame()
    main_event_selected_prior_work_compact = pd.DataFrame()
    main_event_selected_citing_paper_summary = pd.DataFrame()
    for output in main_event_outputs:
        main_event_summary, fig, selected_examples = build_main_text_event_study_figure(
            balanced_event_rows,
            value_col=output["value_col"],
            value_scale=output["value_scale"],
            ylabel=output["ylabel"],
            seed_offset=output["seed_offset"],
            include_selected_examples=output["include_selected_examples"],
            example_source=panel_full,
        )
        main_event_summary = main_event_summary[required_columns].copy()
        main_event_summary.to_csv(output["summary_out"], index=False)
        print(f"wrote {output['summary_out'].relative_to(PROJECT)}")

        fig.savefig(output["figure_out"], bbox_inches="tight")
        fig.savefig(output["report_figure_out"], bbox_inches="tight")
        print(f"wrote {output['figure_out'].relative_to(PROJECT)}")
        print(f"wrote {output['report_figure_out'].relative_to(PROJECT)}")
        plt.close(fig)
        main_event_summaries.append(main_event_summary)

    main_event_selected_examples = build_selected_large_jump_examples(
        panel_full,
        author_metrics,
    )
    if not main_event_selected_examples.empty:
        main_event_selected_citation_sources = (
            build_selected_citation_source_table(main_event_selected_examples)
        )
        main_event_selected_prior_work_summary = (
            build_selected_prior_work_summary(
                main_event_selected_citation_sources
            )
        )
        main_event_selected_prior_work_compact = (
            build_selected_prior_work_compact_table(
                main_event_selected_citation_sources
            )
        )
        main_event_selected_citing_paper_summary = (
            build_selected_citing_paper_summary(
                main_event_selected_citation_sources
            )
        )

    if not main_event_selected_examples.empty:
        main_event_selected_examples.to_csv(
            MAIN_EVENT_SELECTED_LARGE_JUMPS_OUT, index=False
        )
        print(f"wrote {MAIN_EVENT_SELECTED_LARGE_JUMPS_OUT.relative_to(PROJECT)}")
    if not main_event_selected_citation_sources.empty:
        main_event_selected_citation_sources.to_csv(
            MAIN_EVENT_SELECTED_CITATION_SOURCES_OUT, index=False
        )
        print(
            "wrote "
            f"{MAIN_EVENT_SELECTED_CITATION_SOURCES_OUT.relative_to(PROJECT)}"
        )
    if not main_event_selected_prior_work_summary.empty:
        main_event_selected_prior_work_summary.to_csv(
            MAIN_EVENT_SELECTED_PRIOR_WORK_SUMMARY_OUT, index=False
        )
        print(
            "wrote "
            f"{MAIN_EVENT_SELECTED_PRIOR_WORK_SUMMARY_OUT.relative_to(PROJECT)}"
        )
    if not main_event_selected_prior_work_compact.empty:
        main_event_selected_prior_work_compact.to_csv(
            MAIN_EVENT_SELECTED_PRIOR_WORK_COMPACT_OUT, index=False
        )
        print(
            "wrote "
            f"{MAIN_EVENT_SELECTED_PRIOR_WORK_COMPACT_OUT.relative_to(PROJECT)}"
        )
    if not main_event_selected_citing_paper_summary.empty:
        main_event_selected_citing_paper_summary.to_csv(
            MAIN_EVENT_SELECTED_CITING_PAPER_SUMMARY_OUT, index=False
        )
        print(
            "wrote "
            f"{MAIN_EVENT_SELECTED_CITING_PAPER_SUMMARY_OUT.relative_to(PROJECT)}"
        )

    display(pd.concat(main_event_summaries, ignore_index=True))
    if not main_event_selected_examples.empty:
        display(main_event_selected_examples)
    if not main_event_selected_prior_work_compact.empty:
        selected_columns = [
            "name",
            "pc_year",
            "event_time",
            "year",
            "cited_work_title",
            "cited_publication_years",
            "cited_work_age_years",
            "unique_citing_papers",
            "reference_edges",
            "direct_self_edges",
            "coauthor_mediated_edges",
            "citing_papers",
            "citing_authors",
        ]
        display(
            main_event_selected_prior_work_compact.loc[
                main_event_selected_prior_work_compact["event_time"].eq(0),
                selected_columns,
            ]
        )
    if not main_event_selected_citing_paper_summary.empty:
        selected_columns = [
            "name",
            "pc_year",
            "event_time",
            "year",
            "citing_title",
            "citing_authors",
            "cited_prior_works",
            "reference_edges",
            "direct_self_edges",
            "coauthor_mediated_edges",
        ]
        display(
            main_event_selected_citing_paper_summary.loc[
                main_event_selected_citing_paper_summary["event_time"].eq(0),
                selected_columns,
            ]
        )
    if not main_event_selected_citation_sources.empty:
        display(
            main_event_selected_citation_sources[
                [
                    "name",
                    "pc_year",
                    "event_time",
                    "year",
                    "citing_title",
                    "citing_authors",
                    "referenced_work_id",
                ]
            ].head(20)
        )

wrote step_4_artifacts/summary_tables/icfp_selection_log.csv
wrote step_4_artifacts/figures_pre_pc_mean_baseline_event_study/icfp_log.pdf
wrote report_latex/figures/pre_pc_icfp_log.pdf
wrote step_4_artifacts/summary_tables/icfp_selection_raw.csv
wrote step_4_artifacts/figures_pre_pc_mean_baseline_event_study/icfp_raw.pdf
wrote report_latex/figures/pre_pc_icfp_raw.pdf
wrote step_4_artifacts/summary_tables/icfp_selected_jumps.csv
wrote step_4_artifacts/summary_tables/icfp_selected_sources.csv
wrote step_4_artifacts/summary_tables/icfp_selected_prior_summary.csv
wrote step_4_artifacts/summary_tables/icfp_selected_prior_work.csv
wrote step_4_artifacts/summary_tables/icfp_selected_citing_papers.csv


,conference,sample,sample_label,value_scale,event_time,mean,ci_lower,ci_upper,n_event_units
0,ICFP,all_isolated_balanced_event_units,All isolated balanced events,log10_citations_plus_one,-2,-0.016997,-0.053386,0.018000,106
1,ICFP,all_isolated_balanced_event_units,All isolated balanced events,log10_citations_plus_one,-1,0.016997,-0.018000,0.053386,106
2,ICFP,all_isolated_balanced_event_units,All isolated balanced events,log10_citations_plus_one,0,0.082709,0.023922,0.144153,106
3,ICFP,all_isolated_balanced_event_units,All isolated balanced events,log10_citations_plus_one,1,0.025469,-0.047923,0.092110,106
4,ICFP,all_isolated_balanced_event_units,All isolated balanced events,log10_citations_plus_one,2,0.030813,-0.034259,0.094456,106
5,ICFP,all_isolated_balanced_career_age_20_plus,"All isolated balanced events, career age 20+ y...",log10_citations_plus_one,-2,-0.032777,-0.100103,0.026557,39
6,ICFP,all_isolated_balanced_career_age_20_plus,"All isolated balanced events, career age 20+ y...",log10_citations_plus_one,-1,0.032777,-0.026557,0.100103,39
7,ICFP,all_isolated_balanced_career_age_20_plus,"All isolated balanced events, career age 20+ y...",log10_citations_plus_one,0,0.026587,-0.078784,0.135069,39
8,ICFP,all_isolated_balanced_career_age_20_plus,"All isolated balanced events, career age 20+ y...",log10_citations_plus_one,1,-0.040161,-0.138468,0.055107,39
9,ICFP,all_isolated_balanced_career_age_20_plus,"All isolated balanced events, career age 20+ y...",log10_citations_plus_one,2,-0.108751,-0.204591,-0.008927,39


,selection_rank,researcher_id,name,conference,pc_year,citation_t_minus_2,citation_t_minus_1,citation_t_0,citation_t_plus_1,citation_t_plus_2,...,career_age_at_pc_year,career_age_label,career_age_publication_year,career_age_publication_year_status,h_index,has_full_pre_window,has_full_event_window,has_followup_pc_service_t1_t2,same_conference_pc_years,example_note
0,1,ezgicicek,Ezgi Çiçek,ICFP,2019,1.0,5.0,7.0,2.0,0.0,...,4.0,career age 4,2015.0,usable_1980_or_later,5,True,True,False,2019,"Balanced example from the no-earlier-service, ..."
1,2,malgorzatabiernacka,Malgorzata Biernacka,ICFP,2022,0.0,0.0,4.0,1.0,0.0,...,NaN,career age n/a,NaN,excluded_pre_1980_openalex_noise,11,True,True,False,2022,Illustrative raw-count example; earlier broad-...
2,3,peterthiemann,Peter Thiemann,ICFP,2023,3.0,2.0,8.0,1.0,4.0,...,33.0,career age 33,1990.0,usable_1980_or_later,26,True,True,True,2023; 2025,Illustrative raw-count example added for the s...
3,4,martinelsman,Martin Elsman,ICFP,2018,NaN,0.0,7.0,3.0,2.0,...,20.0,career age 20,1998.0,usable_1980_or_later,14,False,False,False,2018; 2024,Illustrative first-PC example; later 2024 PC s...


,name,pc_year,event_time,year,cited_work_title,cited_publication_years,cited_work_age_years,unique_citing_papers,reference_edges,direct_self_edges,coauthor_mediated_edges,citing_papers,citing_authors
2,Ezgi Çiçek,2019,0,2019,NaN,"2015, 2016, 2017","2, 3, 4",2,7,0,5,Closure conversion is safe for space | Relatio...,Weihao Qu; Marco Gaboardi; Deepak Garg | Zoe P...
4,Malgorzata Biernacka,2022,0,2022,NaN,"2007, 2017, 2019, 2021","1, 3, 5, 15",1,4,4,4,A simple and efficient implementation of stron...,Małgorzata Biernacka; Witold Charatonik; Tomas...
6,Martin Elsman,2018,0,2018,NaN,"1999, 2014, 2016, 2017, 2018","0, 1, 2, 4, 19",1,7,7,7,Static interpretation of higher-order modules ...,Martin Elsman; Troels Henriksen; Danil Annenko...
11,Peter Thiemann,2023,0,2023,NaN,"1999, 2004, 2009, 2016, 2020, 2022","1, 3, 7, 14, 19, 24",3,8,4,4,Combinator-Based Fixpoint Algorithms for Big-S...,Jules Jacobs; Jonas Kastberg Hinrichsen; Robbe...


,name,pc_year,event_time,year,citing_title,citing_authors,cited_prior_works,reference_edges,direct_self_edges,coauthor_mediated_edges
3,Ezgi Çiçek,2019,0,2019,Closure conversion is safe for space,Zoe Paraskevopoulou; Andrew W. Appel,"nan (2015, 2016; E=3)",3,0,1
4,Ezgi Çiçek,2019,0,2019,Relational cost analysis for functional-impera...,Weihao Qu; Marco Gaboardi; Deepak Garg,"nan (2016, 2017; E=4)",4,0,4
7,Malgorzata Biernacka,2022,0,2022,A simple and efficient implementation of stron...,Małgorzata Biernacka; Witold Charatonik; Tomas...,"nan (2007, 2017, 2019, 2021; E=4)",4,4,4
9,Martin Elsman,2018,0,2018,Static interpretation of higher-order modules ...,Martin Elsman; Troels Henriksen; Danil Annenko...,"nan (1999, 2014, 2016, 2017, 2018; E=7)",7,7,7
19,Peter Thiemann,2023,0,2023,Combinator-Based Fixpoint Algorithms for Big-S...,Sven Keidel; Sebastian Erdweg; Tobias Hombücher,nan (2009; E=1),1,0,0
20,Peter Thiemann,2023,0,2023,Dependent Session Protocols in Separation Logi...,Jules Jacobs; Jonas Kastberg Hinrichsen; Robbe...,"nan (2020, 2022; E=3)",3,0,0
21,Peter Thiemann,2023,0,2023,Intrinsically Typed Sessions with Callbacks (F...,Peter Thiemann,"nan (1999, 2004, 2016, 2022; E=4)",4,4,4


,name,pc_year,event_time,year,citing_title,citing_authors,referenced_work_id
0,Ezgi Çiçek,2019,-2,2017,A relational logic for higher-order programs,Alejandro Aguirre; Gilles Barthe; Marco Gaboar...,W2565334366
1,Ezgi Çiçek,2019,-1,2018,Casts and costs: harmonizing safety and perfor...,John Peter Campora; Sheng Chen; Eric Walkingshaw,W2514317887
2,Ezgi Çiçek,2019,-1,2018,Casts and costs: harmonizing safety and perfor...,John Peter Campora; Sheng Chen; Eric Walkingshaw,W2565334366
3,Ezgi Çiçek,2019,-1,2018,Casts and costs: harmonizing safety and perfor...,John Peter Campora; Sheng Chen; Eric Walkingshaw,W3004472005
4,Ezgi Çiçek,2019,-1,2018,Casts and costs: harmonizing safety and perfor...,John Peter Campora; Sheng Chen; Eric Walkingshaw,W3014043685
5,Ezgi Çiçek,2019,-1,2018,Parallel complexity analysis with temporal ses...,Ankush Das; Jan Hoffmann; Frank Pfenning,W2565334366
6,Ezgi Çiçek,2019,0,2019,Closure conversion is safe for space,Zoe Paraskevopoulou; Andrew W. Appel,W2119859640
7,Ezgi Çiçek,2019,0,2019,Closure conversion is safe for space,Zoe Paraskevopoulou; Andrew W. Appel,W2514317887
8,Ezgi Çiçek,2019,0,2019,Closure conversion is safe for space,Zoe Paraskevopoulou; Andrew W. Appel,W2565334366
9,Ezgi Çiçek,2019,0,2019,Relational cost analysis for functional-impera...,Weihao Qu; Marco Gaboardi; Deepak Garg,W2514317887


## 9. Plot selected PC member citation trajectories

In [9]:
def build_spike_fade_cases(rows, n_cases=6):
    wide = (
        rows.pivot_table(
            index=[
                "event_unit_id",
                "researcher_id",
                "name",
                "conference",
                "first_pc_year_conference",
                "career_age_at_first_pc",
                "source_history_pc_group_this_conference",
            ],
            columns="event_time",
            values="citation_count",
            aggfunc="first",
        )
        .reindex(columns=EVENT_TIMES)
        .reset_index()
    )
    wide = wide.rename(
        columns={
            -2: "citation_t_minus_2",
            -1: "citation_t_minus_1",
            0: "citation_t_0",
            1: "citation_t_plus_1",
            2: "citation_t_plus_2",
        }
    )
    citation_cols = [
        "citation_t_minus_2",
        "citation_t_minus_1",
        "citation_t_0",
        "citation_t_plus_1",
        "citation_t_plus_2",
    ]
    for col in citation_cols:
        wide[col] = pd.to_numeric(wide[col], errors="coerce")

    wide["baseline_raw_citations"] = (
        wide["citation_t_minus_2"] + wide["citation_t_minus_1"]
    ) / 2
    wide["post_pc_followup_mean_raw"] = (
        wide["citation_t_plus_1"] + wide["citation_t_plus_2"]
    ) / 2
    wide["pc_year_jump_raw"] = (
        wide["citation_t_0"] - wide["baseline_raw_citations"]
    )
    wide["fade_t0_to_post_mean_raw"] = (
        wide["citation_t_0"] - wide["post_pc_followup_mean_raw"]
    )
    wide["followup_both_below_pc_year"] = (
        wide["citation_t_plus_1"].lt(wide["citation_t_0"])
        & wide["citation_t_plus_2"].lt(wide["citation_t_0"])
    )
    wide["spike_and_fade_raw"] = (
        wide["pc_year_jump_raw"].gt(0)
        & wide["fade_t0_to_post_mean_raw"].gt(0)
        & wide["followup_both_below_pc_year"]
    )
    wide["below_baseline_by_post_mean"] = (
        wide["spike_and_fade_raw"]
        & wide["post_pc_followup_mean_raw"].le(
            wide["baseline_raw_citations"]
        )
    )

    selected = (
        wide.loc[wide["spike_and_fade_raw"]]
        .sort_values(
            [
                "pc_year_jump_raw",
                "fade_t0_to_post_mean_raw",
                "citation_t_0",
            ],
            ascending=False,
        )
        .head(n_cases)
        .copy()
    )
    selected.insert(0, "selection_rank", range(1, len(selected) + 1))
    return wide, selected


def selected_pc_member_specs():
    return [
        {
            "panel_order": 1,
            "researcher_id": "clementpitclaudel",
            "name": "Clément Pit-Claudel",
            "events": [
                {"conference": "PLDI", "pc_year": 2024, "label": "PLDI 2024"},
                {
                    "conference": "OOPSLA_COMBINED",
                    "pc_year": 2025,
                    "label": "OOPSLA 2025",
                },
            ],
        },
        {
            "panel_order": 2,
            "researcher_id": "viktorkuncak",
            "name": "Viktor Kunčak",
            "events": [
                {"conference": "OOPSLA", "pc_year": 2020, "label": "OOPSLA 2020"},
                {"conference": "PLDI", "pc_year": 2021, "label": "PLDI 2021"},
            ],
        },
        {
            "panel_order": 3,
            "researcher_id": "peterthiemann",
            "name": "Peter Thiemann",
            "events": [
                {"conference": "ICFP", "pc_year": 2023, "label": "ICFP 2023"},
            ],
        },
        {
            "panel_order": 4,
            "researcher_id": "simonjgay",
            "name": "Simon J. Gay",
            "events": [
                {"conference": "ICFP", "pc_year": 2023, "label": "ICFP 2023"},
            ],
        },
        {
            "panel_order": 5,
            "researcher_id": "martinelsman",
            "name": "Martin Elsman",
            "events": [
                {"conference": "ICFP", "pc_year": 2024, "label": "ICFP 2024"},
            ],
        },
        {
            "panel_order": 6,
            "researcher_id": "ezgicicek",
            "name": "Ezgi Çiçek",
            "events": [
                {"conference": "ICFP", "pc_year": 2019, "label": "ICFP 2019"},
            ],
        },
        {
            "panel_order": 7,
            "researcher_id": "brandonlucia",
            "name": "Brandon Lucia",
            "events": [
                {"conference": "OOPSLA", "pc_year": 2019, "label": "OOPSLA 2019"},
            ],
        },
        {
            "panel_order": 8,
            "researcher_id": "jonathanbrachthauser",
            "name": "Jonathan Immanuel Brachthäuser",
            "events": [
                {"conference": "POPL", "pc_year": 2024, "label": "POPL 2024"},
            ],
        },
        {
            "panel_order": 9,
            "researcher_id": "fritzhenglein",
            "name": "Fritz Henglein",
            "events": [
                {"conference": "POPL", "pc_year": 2024, "label": "POPL 2024"},
            ],
        },
        {
            "panel_order": 10,
            "researcher_id": "malgorzatabiernacka",
            "name": "Malgorzata Biernacka",
            "events": [
                {"conference": "ICFP", "pc_year": 2022, "label": "ICFP 2022"},
            ],
        },
    ]


OPENALEX_CAREER_AGE_OVERRIDES = {
    "malgorzatabiernacka": 18,
}


GOOGLE_SCHOLAR_FIRST_CITATION_YEAR = {
    "brandonlucia": 2009,
    "clementpitclaudel": 2016,
    "ezgicicek": 2014,
    "jonathanbrachthauser": 2014,
    "malgorzatabiernacka": 2004,
    "martinelsman": 1997,
    "peterthiemann": 1995,
    "fritzhenglein": 1991,
    "simonjgay": 1994,
    "viktorkuncak": 2003,
}


GOOGLE_SCHOLAR_SCREENSHOT_FILES = {
    "brandonlucia": "Brandon Lucia.png",
    "clementpitclaudel": "Clement_.png",
    "ezgicicek": "Ezgi_Çiçek_Career_Age_ICFP.png",
    "jonathanbrachthauser": "Jonathan Immanuel Brachthäuser .png",
    "malgorzatabiernacka": "Malgorzata Biernacka_career_age_ICFP.png",
    "martinelsman": "Martin_Elsman_career_age_ICFP.png",
    "peterthiemann": "Peter_thieman.png",
    "fritzhenglein": "fritz_henglein.png",
    "simonjgay": "simon_j_gay.png",
    "viktorkuncak": "victor_kuncak.png",
}


GOOGLE_SCHOLAR_REPORT_SCREENSHOT_FILES = {
    "clementpitclaudel": "clement_pit_claudel.png",
    "viktorkuncak": "viktor_kuncak.png",
    "peterthiemann": "peter_thiemann.png",
    "simonjgay": "simon_j_gay.png",
    "martinelsman": "martin_elsman.png",
    "ezgicicek": "ezgi_cicek.png",
    "brandonlucia": "brandon_lucia.png",
    "jonathanbrachthauser": "jonathan_brachthaeuser.png",
    "fritzhenglein": "fritz_henglein.png",
    "malgorzatabiernacka": "malgorzata_biernacka.png",
}


GOOGLE_SCHOLAR_CHART_CROP_BOXES = {
    "Brandon Lucia.png": (1078, 704, 2365, 1344),
    "Clement_.png": (1310, 674, 2141, 1315),
    "Ezgi_Çiçek_Career_Age_ICFP.png": (1240, 668, 2201, 1308),
    "Jonathan Immanuel Brachthäuser .png": (1237, 710, 2195, 1348),
    "Malgorzata Biernacka_career_age_ICFP.png": (900, 740, 2540, 1560),
    "Martin_Elsman_career_age_ICFP.png": (697, 684, 2746, 1319),
    "Peter_thieman.png": (640, 720, 2880, 1350),
    "fritz_henglein.png": (413, 703, 3034, 1337),
    "simon_j_gay.png": (592, 710, 2833, 1348),
    "victor_kuncak.png": (870, 702, 2556, 1342),
}


GOOGLE_SCHOLAR_NODE_FILES = {
    "brandonlucia": "Brandon_Lucia_node.png",
    "clementpitclaudel": "Clement_Pit_Claudel_node.png",
    "ezgicicek": "Ezgi_Cicek_node.png",
    "fritzhenglein": "Fritz_Henglein_node.png",
    "jonathanbrachthauser": "Jonathan_Brachthauser_node.png",
    "malgorzatabiernacka": "Malgorzata_Biernacka_node.png",
    "martinelsman": "Martin_Elsman_node.png",
    "peterthiemann": "Peter_Thiemann_node.png",
    "simonjgay": "Simon_J_Gay_node.png",
    "viktorkuncak": "Viktor_Kuncak_node.png",
}


GOOGLE_SCHOLAR_SCREENSHOT_DATE = "June 13, 2026"


def selected_event_rows(panel_source, researcher_id, conference, pc_year):
    years = [pc_year + event_time for event_time in EVENT_TIMES]
    base = panel_source.loc[
        panel_source["researcher_id"].eq(researcher_id)
        & panel_source["year"].isin(years)
    ].copy()
    if conference == "OOPSLA_COMBINED":
        base = base.loc[base["conference"].isin(["OOPSLA1", "OOPSLA2"])].copy()
        grouped = (
            base.groupby(["researcher_id", "name", "year"], as_index=False)
            .agg(
                citation_count=("citation_count", "sum"),
                pc_status=("pc_status", "max"),
            )
        )
        grouped["conference"] = "OOPSLA combined"
        return grouped

    return base.loc[base["conference"].eq(conference)].copy()


def build_selected_pc_member_trajectory_cases(panel_source, author_metric_source):
    records = []
    author_metric_lookup = (
        author_metric_source.drop_duplicates("researcher_id")
        .set_index("researcher_id")
        .to_dict("index")
    )
    for spec in selected_pc_member_specs():
        metrics = author_metric_lookup.get(spec["researcher_id"], {})
        career_year = metrics.get("career_age_publication_year", np.nan)
        for line_order, event in enumerate(spec["events"], start=1):
            event_rows = selected_event_rows(
                panel_source,
                researcher_id=spec["researcher_id"],
                conference=event["conference"],
                pc_year=event["pc_year"],
            )
            event_lookup = event_rows.set_index("year").to_dict("index")
            row = {
                "selection_rank": spec["panel_order"],
                "panel_order": spec["panel_order"],
                "line_order": line_order,
                "researcher_id": spec["researcher_id"],
                "name": spec["name"],
                "conference": event["conference"],
                "plot_conference": (
                    "OOPSLA combined"
                    if event["conference"] == "OOPSLA_COMBINED"
                    else event["conference"]
                ),
                "pc_year": event["pc_year"],
                "selected_event_label": event["label"],
                "selected_trajectory_version": "selected_researchers_final_no_story_nodes",
                "google_scholar_screenshot_date": GOOGLE_SCHOLAR_SCREENSHOT_DATE,
            }
            for event_time in EVENT_TIMES:
                year = event["pc_year"] + event_time
                value = event_lookup.get(year, {}).get("citation_count", np.nan)
                col = {
                    -2: "citation_t_minus_2",
                    -1: "citation_t_minus_1",
                    0: "citation_t_0",
                    1: "citation_t_plus_1",
                    2: "citation_t_plus_2",
                }[event_time]
                row[col] = value
            override_age = OPENALEX_CAREER_AGE_OVERRIDES.get(spec["researcher_id"])
            if override_age is not None:
                career_age = override_age
                row["openalex_career_age_at_pc_year"] = career_age
                row["career_age_at_pc_year"] = career_age
                row["career_age_label"] = f"career age {career_age}"
                row["career_age_publication_year"] = event["pc_year"] - career_age
                row["career_age_publication_year_status"] = "manual_override"
            elif pd.notna(career_year):
                career_age = event["pc_year"] - int(career_year)
                row["openalex_career_age_at_pc_year"] = career_age
                row["career_age_at_pc_year"] = career_age
                row["career_age_label"] = f"career age {career_age}"
                row["career_age_publication_year"] = career_year
                row["career_age_publication_year_status"] = metrics.get(
                    "career_age_publication_year_status", pd.NA
                )
            else:
                career_age = np.nan
                row["openalex_career_age_at_pc_year"] = np.nan
                row["career_age_at_pc_year"] = np.nan
                row["career_age_label"] = "career age n/a"
                row["career_age_publication_year"] = np.nan
                row["career_age_publication_year_status"] = pd.NA

            gs_first_year = GOOGLE_SCHOLAR_FIRST_CITATION_YEAR.get(
                spec["researcher_id"], np.nan
            )
            row["google_scholar_first_citation_year"] = gs_first_year
            row["google_scholar_career_age_at_pc_year"] = (
                event["pc_year"] - int(gs_first_year)
                if pd.notna(gs_first_year)
                else np.nan
            )
            row["google_scholar_screenshot_file"] = (
                GOOGLE_SCHOLAR_SCREENSHOT_FILES.get(spec["researcher_id"], pd.NA)
            )
            row["selected_legend_label"] = event["label"]
            records.append(row)

    selected = pd.DataFrame(records)
    selected["baseline_raw_citations"] = selected[
        ["citation_t_minus_2", "citation_t_minus_1"]
    ].mean(axis=1, skipna=True)
    selected["pc_year_jump_raw"] = (
        selected["citation_t_0"] - selected["baseline_raw_citations"]
    )
    selected["jump_t_minus_1_to_t0_raw"] = (
        selected["citation_t_0"] - selected["citation_t_minus_1"]
    )
    selected["fade_t0_to_t_plus_1_raw"] = (
        selected["citation_t_plus_1"] - selected["citation_t_0"]
    )
    selected["spike_and_fade_raw"] = (
        selected["jump_t_minus_1_to_t0_raw"].gt(0)
        & selected["fade_t0_to_t_plus_1_raw"].lt(0)
    )
    selected["marker_definition"] = "trajectory_only"
    selected["figure_annotation_mode"] = "multi_researcher_selected_trajectories"
    return selected.sort_values(["panel_order", "line_order"]).reset_index(drop=True)


def crop_google_scholar_chart(image, screenshot_name):
    crop_box = GOOGLE_SCHOLAR_CHART_CROP_BOXES.get(screenshot_name)
    if crop_box is None:
        width, height = image.size
        crop_box = (
            int(width * 0.22),
            int(height * 0.33),
            int(width * 0.75),
            int(height * 0.76),
        )
    return image.crop(crop_box)


def draw_google_scholar_screenshot_grid():
    representative_specs = selected_pc_member_specs()

    ncols = 5
    nrows = 2
    thumb_width = 1250
    thumb_height = 540
    pad = 45
    label_height = 64
    canvas_width = ncols * thumb_width + (ncols + 1) * pad
    canvas_height = nrows * (thumb_height + label_height) + (nrows + 1) * pad
    canvas = Image.new("RGB", (canvas_width, canvas_height), "white")
    draw = ImageDraw.Draw(canvas)
    try:
        label_font = ImageFont.truetype(
            "/System/Library/Fonts/Supplemental/Arial.ttf", 22
        )
        small_font = ImageFont.truetype(
            "/System/Library/Fonts/Supplemental/Arial.ttf", 34
        )
    except OSError:
        label_font = None
        small_font = None

    for index, spec in enumerate(representative_specs):
        col = index % ncols
        row = index // ncols
        x = pad + col * (thumb_width + pad)
        y = pad + row * (thumb_height + label_height + pad)
        screenshot_name = GOOGLE_SCHOLAR_SCREENSHOT_FILES.get(
            spec["researcher_id"]
        )
        screenshot_path = (
            GS_SCREENSHOT_DIR / screenshot_name
            if screenshot_name is not None
            else None
        )
        if screenshot_path is not None and screenshot_path.exists():
            crop = crop_google_scholar_chart(
                Image.open(screenshot_path).convert("RGB"),
                screenshot_name,
            )
            crop.thumbnail((thumb_width, thumb_height), Image.LANCZOS)
            canvas.paste(
                crop,
                (
                    x + (thumb_width - crop.width) // 2,
                    y + (thumb_height - crop.height) // 2,
                ),
            )
        else:
            draw.rectangle(
                [x, y, x + thumb_width, y + thumb_height],
                outline=(160, 160, 160),
                width=2,
            )
            draw.text(
                (x + 30, y + 105),
                "Google Scholar\nscreenshot not available",
                fill=(80, 80, 80),
                font=label_font,
            )
        label = spec["name"]
        first_year = GOOGLE_SCHOLAR_FIRST_CITATION_YEAR.get(spec["researcher_id"])
        if first_year is not None:
            label += f"  |  first year: {first_year}"
        draw.text((x, y + thumb_height + 8), label, fill="black", font=small_font)

    return canvas


def copy_google_scholar_screenshots_for_latex():
    copied = []
    for spec in selected_pc_member_specs():
        researcher_id = spec["researcher_id"]
        source_name = GOOGLE_SCHOLAR_SCREENSHOT_FILES.get(researcher_id)
        output_name = GOOGLE_SCHOLAR_REPORT_SCREENSHOT_FILES.get(researcher_id)
        if source_name is None or output_name is None:
            continue
        source_path = GS_SCREENSHOT_DIR / source_name
        output_path = REPORT_GS_SCREENSHOT_DIR / output_name
        if not source_path.exists():
            raise FileNotFoundError(source_path)
        if setup.overwrite_artifacts or not output_path.exists():
            output_path.write_bytes(source_path.read_bytes())
        copied.append(output_path)
    return copied


def make_black_background_transparent(image):
    image = image.convert("RGBA")
    pixels = np.asarray(image).copy()
    black_background = (
        (pixels[:, :, 0] < 8)
        & (pixels[:, :, 1] < 8)
        & (pixels[:, :, 2] < 8)
    )
    pixels[black_background, 3] = 0
    return Image.fromarray(pixels, mode="RGBA")


def story_node_high_res_path(researcher_id):
    node_file = GOOGLE_SCHOLAR_NODE_FILES.get(researcher_id)
    if node_file is None:
        return None
    node_name = Path(node_file)
    return GS_NODE_HIGH_RES_DIR / f"{node_name.stem}_hires{node_name.suffix}"


def build_high_res_story_node_images():
    high_res_paths = []
    for researcher_id, node_file in GOOGLE_SCHOLAR_NODE_FILES.items():
        source_path = GS_NODE_DIR / node_file
        output_path = story_node_high_res_path(researcher_id)
        if output_path is None or not source_path.exists():
            continue

        source_is_newer = (
            output_path.exists()
            and source_path.stat().st_mtime > output_path.stat().st_mtime
        )
        if setup.overwrite_artifacts or not output_path.exists() or source_is_newer:
            image = make_black_background_transparent(Image.open(source_path))
            high_res_size = (
                image.width * STORY_NODE_SCALE,
                image.height * STORY_NODE_SCALE,
            )
            image = image.resize(high_res_size, Image.Resampling.LANCZOS)
            image = image.filter(
                ImageFilter.UnsharpMask(radius=1.1, percent=90, threshold=3)
            )
            image.save(
                output_path,
                dpi=(STORY_NODE_EXPORT_DPI, STORY_NODE_EXPORT_DPI),
                optimize=True,
            )
        high_res_paths.append(output_path)
    return high_res_paths


def load_story_node_image(researcher_id):
    node_file = GOOGLE_SCHOLAR_NODE_FILES.get(researcher_id)
    if node_file is None:
        return None
    high_res_path = story_node_high_res_path(researcher_id)
    node_path = (
        high_res_path
        if high_res_path is not None and high_res_path.exists()
        else GS_NODE_DIR / node_file
    )
    if not node_path.exists():
        return None
    image = make_black_background_transparent(Image.open(node_path))
    return np.asarray(image)


def add_story_node_to_axis(ax, researcher_id):
    node_image = load_story_node_image(researcher_id)
    if node_image is None:
        return
    image_box = OffsetImage(
        node_image,
        zoom=0.02625,
        interpolation="lanczos",
        dpi_cor=True,
    )
    transform = blended_transform_factory(ax.transData, ax.transAxes)
    annotation = AnnotationBbox(
        image_box,
        (1.55, 0.80),
        xycoords=transform,
        frameon=False,
        box_alignment=(0.5, 0.5),
        pad=0,
        zorder=2,
    )
    ax.add_artist(annotation)


def build_selected_event_study_check_rows(selected):
    records = []
    citation_columns = {
        -2: "citation_t_minus_2",
        -1: "citation_t_minus_1",
        0: "citation_t_0",
        1: "citation_t_plus_1",
        2: "citation_t_plus_2",
    }
    for _, row in selected.iterrows():
        baseline_raw = row["citation_t_minus_2"]
        baseline_log = (
            np.log10(baseline_raw + 1)
            if pd.notna(baseline_raw)
            else np.nan
        )
        for event_time, citation_col in citation_columns.items():
            citation_count = row[citation_col]
            log_value = (
                np.log10(citation_count + 1)
                if pd.notna(citation_count)
                else np.nan
            )
            records.append(
                {
                    "selection_rank": row["selection_rank"],
                    "panel_order": row["panel_order"],
                    "line_order": row["line_order"],
                    "researcher_id": row["researcher_id"],
                    "name": row["name"],
                    "conference": row["conference"],
                    "plot_conference": row["plot_conference"],
                    "pc_year": row["pc_year"],
                    "selected_event_label": row["selected_event_label"],
                    "event_time": event_time,
                    "year": row["pc_year"] + event_time,
                    "citation_count": citation_count,
                    "log10_citations_plus_one": log_value,
                    "delta_log10_t_minus_2_reference": (
                        log_value - baseline_log
                        if pd.notna(log_value) and pd.notna(baseline_log)
                        else np.nan
                    ),
                    "delta_raw_t_minus_2_reference": (
                        citation_count - baseline_raw
                        if pd.notna(citation_count) and pd.notna(baseline_raw)
                        else np.nan
                    ),
                    "openalex_career_age_at_pc_year": row[
                        "openalex_career_age_at_pc_year"
                    ],
                    "google_scholar_career_age_at_pc_year": row[
                        "google_scholar_career_age_at_pc_year"
                    ],
                }
            )
    return pd.DataFrame(records)


def build_selected_event_study_mean_summary(
    check_rows,
    value_col="delta_log10_t_minus_2_reference",
    value_scale="log10_citations_plus_one_t_minus_2_reference",
    seed=1234,
):
    matrix = (
        check_rows.pivot_table(
            index=[
                "researcher_id",
                "conference",
                "pc_year",
                "selected_event_label",
            ],
            columns="event_time",
            values=value_col,
            aggfunc="first",
        )
        .reindex(columns=EVENT_TIMES)
        .to_numpy(dtype=float)
    )
    summary = bootstrap_summary(matrix, EVENT_TIMES, seed=seed)
    summary.insert(0, "value_scale", value_scale)
    summary["bootstrap_unit"] = "selected_pc_service_trajectory"
    summary["bootstrap_reps"] = BOOTSTRAP_REPS
    summary["n_non_missing"] = np.sum(~np.isnan(matrix), axis=0)
    return summary


def draw_selected_event_study_check(
    selected,
    check_rows,
    mean_summary,
    background_rows,
):
    if selected.empty or check_rows.empty:
        raise ValueError("No selected PC-member event-study rows available.")

    soft_black = "0.28"
    reference_gray = "0.35"
    fig = plt.figure(figsize=(9.2, 6.2))
    grid = fig.add_gridspec(
        nrows=2,
        ncols=2,
        height_ratios=[1.28, 1.0],
        width_ratios=[1.45, 0.95],
        hspace=0.55,
        wspace=0.30,
    )
    ax_event = fig.add_subplot(grid[0, :])
    ax_raw = fig.add_subplot(grid[1, 0])
    ax_age = fig.add_subplot(grid[1, 1])

    event_times = np.array(EVENT_TIMES)
    x_tick_labels = ["-2", "-1", "0\nPC", "+1", "+2"]
    selected_keys = (
        selected[
            [
                "panel_order",
                "line_order",
                "researcher_id",
                "selected_event_label",
                "plot_conference",
            ]
        ]
        .sort_values(["panel_order", "line_order"])
        .reset_index(drop=True)
    )

    for _, key in selected_keys.iterrows():
        trajectory = check_rows.loc[
            check_rows["researcher_id"].eq(key["researcher_id"])
            & check_rows["selected_event_label"].eq(
                key["selected_event_label"]
            )
        ].sort_values("event_time")
        color = CONFERENCE_COLORS.get(key["plot_conference"], MAIN_COLOR)
        ax_event.scatter(
            trajectory["event_time"],
            trajectory["delta_log10_t_minus_2_reference"],
            color=color,
            s=18,
            edgecolors=soft_black,
            linewidths=0.25,
            alpha=0.70,
            zorder=2,
        )

    mean_trajectory = mean_summary.sort_values("event_time").copy()
    mean_post = mean_trajectory.loc[mean_trajectory["event_time"].ge(0)].copy()
    if not mean_post.empty:
        mean_yerr = np.vstack(
            [
                mean_post["mean"] - mean_post["ci_lower"],
                mean_post["ci_upper"] - mean_post["mean"],
            ]
        )
        ax_event.errorbar(
            mean_post["event_time"],
            mean_post["mean"],
            yerr=mean_yerr,
            fmt="none",
            ecolor=soft_black,
            elinewidth=0.95,
            capsize=2.5,
            capthick=0.95,
            zorder=5,
        )
    ax_event.errorbar(
        mean_trajectory["event_time"],
        mean_trajectory["mean"],
        color=soft_black,
        fmt="D",
        markersize=4.2,
        label="mean",
        zorder=5,
    )
    ax_event.axhline(0, color=reference_gray, lw=0.75, alpha=0.58)
    ax_event.axvline(0, color=reference_gray, lw=0.8, ls="--", alpha=0.58)
    ax_event.set_xlim(EVENT_TIMES.min() - 0.25, EVENT_TIMES.max() + 0.25)
    ax_event.set_xticks(event_times)
    ax_event.set_xticklabels(x_tick_labels)
    ax_event.set_ylabel(
        r"$\Delta \log_{10}(\mathrm{citations}+1)$"
    )
    ax_event.set_title(
        r"(a) Citation change from $t=-2$",
        loc="left",
        fontsize=10.3,
    )
    ax_event.grid(axis="y", color="0.88", lw=0.65, ls=":")
    ax_event.spines["top"].set_visible(False)
    ax_event.spines["right"].set_visible(False)

    legend_handles = [
        plt.Line2D(
            [0],
            [0],
            color=CONFERENCE_COLORS[label],
            marker="o",
            linestyle="none",
            markersize=3.2,
            label=label,
        )
        for label in ["ICFP", "OOPSLA", "PLDI", "POPL"]
    ]
    legend_handles.append(
        plt.Line2D(
            [0],
            [0],
            color=soft_black,
            marker="D",
            linestyle="none",
            markersize=3.6,
            label="mean",
        )
    )
    ax_event.legend(
        handles=legend_handles,
        loc="upper left",
        ncol=5,
        frameon=False,
        fontsize=7.4,
        handlelength=1.35,
        columnspacing=0.9,
    )

    background_units = background_rows.drop_duplicates("event_unit_id")
    background_age = pd.to_numeric(
        background_units["career_age_at_first_pc"], errors="coerce"
    ).dropna()
    selected_age = pd.to_numeric(
        selected["openalex_career_age_at_pc_year"], errors="coerce"
    )
    ax_age.boxplot(
        [background_age.to_numpy()],
        vert=False,
        positions=[1.0],
        widths=0.36,
        patch_artist=True,
        showfliers=False,
        boxprops=dict(facecolor="0.92", edgecolor=soft_black, linewidth=0.8),
        medianprops=dict(color=soft_black, linewidth=1.0),
        whiskerprops=dict(color=soft_black, linewidth=0.8),
        capprops=dict(color=soft_black, linewidth=0.8),
    )
    rng = np.random.default_rng(1234)
    selected_jitter = 1.0 + rng.uniform(-0.10, 0.10, size=len(selected))
    for idx, (_, row) in enumerate(selected.iterrows()):
        age = row["openalex_career_age_at_pc_year"]
        if pd.isna(age):
            continue
        color = CONFERENCE_COLORS.get(row["plot_conference"], MAIN_COLOR)
        ax_age.scatter(
            age,
            selected_jitter[idx],
            color=color,
            edgecolor=soft_black,
            linewidth=0.35,
            s=25,
            zorder=4,
        )
    ax_age.set_yticks([])
    ax_age.set_xlabel("Career age at PC year")
    ax_age.set_title("(c) Career age of selected researchers", loc="left", fontsize=9.7)
    ax_age.grid(axis="x", color="0.88", lw=0.65, ls=":")
    ax_age.spines["top"].set_visible(False)
    ax_age.spines["right"].set_visible(False)
    ax_age.spines["left"].set_visible(False)

    raw_by_time = [
        check_rows.loc[
            check_rows["event_time"].eq(event_time), "citation_count"
        ]
        .dropna()
        .to_numpy()
        for event_time in EVENT_TIMES
    ]
    ax_raw.boxplot(
        raw_by_time,
        positions=event_times,
        widths=0.42,
        patch_artist=True,
        showfliers=False,
        boxprops=dict(facecolor="0.92", edgecolor=soft_black, linewidth=0.8),
        medianprops=dict(color=soft_black, linewidth=1.0),
        whiskerprops=dict(color=soft_black, linewidth=0.8),
        capprops=dict(color=soft_black, linewidth=0.8),
    )
    for _, key in selected_keys.iterrows():
        trajectory = check_rows.loc[
            check_rows["researcher_id"].eq(key["researcher_id"])
            & check_rows["selected_event_label"].eq(
                key["selected_event_label"]
            )
        ].sort_values("event_time")
        color = CONFERENCE_COLORS.get(key["plot_conference"], MAIN_COLOR)
        ax_raw.plot(
            trajectory["event_time"],
            trajectory["citation_count"],
            color=color,
            marker="o",
            markersize=3.0,
            linewidth=0.9,
            alpha=0.55,
            zorder=3,
        )
    raw_mean = (
        check_rows.groupby("event_time", as_index=False)["citation_count"]
        .mean()
        .sort_values("event_time")
    )
    ax_raw.plot(
        raw_mean["event_time"],
        raw_mean["citation_count"],
        color=soft_black,
        marker="D",
        markersize=4.5,
        linewidth=1.6,
        zorder=5,
        label="selected mean",
    )
    ax_raw.axvline(0, color=reference_gray, lw=0.8, ls="--", alpha=0.58)
    ax_raw.set_xticks(event_times)
    ax_raw.set_xticklabels(x_tick_labels)
    ax_raw.set_xlabel("Event time")
    ax_raw.set_ylabel("Raw citation count")
    ax_raw.set_title(
        "(b) Raw citation counts",
        loc="left",
        fontsize=9.7,
    )
    ax_raw.grid(axis="y", color="0.88", lw=0.65, ls=":")
    ax_raw.spines["top"].set_visible(False)
    ax_raw.spines["right"].set_visible(False)

    fig.tight_layout()
    return fig


def draw_spike_fade_examples(selected):
    if selected.empty:
        raise ValueError("No selected PC-member trajectories available to plot.")

    y_cols = [
        "citation_t_minus_2",
        "citation_t_minus_1",
        "citation_t_0",
        "citation_t_plus_1",
        "citation_t_plus_2",
    ]
    panel_keys = (
        selected[["panel_order", "researcher_id", "name"]]
        .drop_duplicates()
        .sort_values("panel_order")
    )
    n_panels = panel_keys.shape[0]
    ncols = 2
    nrows = int(np.ceil(n_panels / ncols))
    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(9.0, 1.85 * nrows),
        sharex=True,
        sharey=False,
    )
    axes = np.array(axes).ravel()
    event_times = np.array(EVENT_TIMES)
    x_tick_labels = ["-2", "-1", "0\nPC", "+1", "+2"]

    for ax, (_, panel_row) in zip(axes, panel_keys.iterrows()):
        panel_data = selected.loc[
            selected["researcher_id"].eq(panel_row["researcher_id"])
        ].sort_values("line_order")
        ax.axvline(0, color="black", lw=0.8, ls="--", alpha=0.75)
        panel_y_max = 0.0
        for _, row in panel_data.iterrows():
            y = np.array([row[col] for col in y_cols], dtype=float)
            if np.isfinite(y).any():
                panel_y_max = max(panel_y_max, float(np.nanmax(y)))
            color = CONFERENCE_COLORS.get(row["plot_conference"], MAIN_COLOR)
            ax.plot(
                event_times,
                y,
                color=color,
                lw=1.6,
                marker="o",
                markersize=3.8,
                markeredgecolor="black",
                markeredgewidth=0.35,
                label=row["selected_legend_label"],
                zorder=3,
            )
        ax.set_title(
            panel_row["name"],
            loc="left",
            fontsize=7.5,
        )
        ax.set_xlim(EVENT_TIMES.min() - 0.25, EVENT_TIMES.max() + 0.25)
        ax.set_ylim(-0.35, max(4, np.ceil((panel_y_max + 1) / 2) * 2))
        ax.yaxis.set_major_locator(MaxNLocator(integer=True, nbins=5))
        ax.set_xticks(event_times)
        ax.set_xticklabels(x_tick_labels)
        ax.set_xlabel("Event time")
        ax.xaxis.label.set_size(8.0)
        ax.yaxis.label.set_size(8.0)
        ax.tick_params(axis="both", labelsize=7.5)
        ax.tick_params(axis="x", labelbottom=True)
        ax.grid(color="#DDDDDD", ls=":", lw=0.7)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.legend(
            loc="upper left",
            frameon=False,
            fontsize=6.3,
            handlelength=1.2,
            labelspacing=0.2,
            borderaxespad=0.2,
        )
    for ax in axes[n_panels:]:
        ax.axis("off")

    axes[0].set_ylabel("Citations")

    fig.tight_layout(w_pad=1.4, h_pad=1.6)
    return fig


if balanced_event_rows.empty:
    print("Skipping selected examples until event-study rows are available.")
else:
    spike_fade_pool, automatic_spike_fade_cases = build_spike_fade_cases(
        balanced_event_rows,
        n_cases=6,
    )
    selected_spike_fade = build_selected_pc_member_trajectory_cases(
        panel_full,
        author_metrics,
    )
    print(f"isolated balanced event units: {spike_fade_pool.shape[0]:,}")
    print(
        "raw spike-and-fade cases: "
        f"{int(spike_fade_pool['spike_and_fade_raw'].sum()):,} "
        f"({spike_fade_pool['spike_and_fade_raw'].mean() * 100:.1f}%)"
    )
    print(
        "positive PC-year jump and follow-up mean below baseline: "
        f"{int(spike_fade_pool['below_baseline_by_post_mean'].sum()):,} "
        f"({spike_fade_pool['below_baseline_by_post_mean'].mean() * 100:.1f}%)"
    )
    display(
        spike_fade_pool.groupby("conference")["spike_and_fade_raw"]
        .agg(["sum", "count"])
        .assign(share=lambda d: d["sum"] / d["count"])
        .reset_index()
    )

    spike_fade_required_columns = [
        "jump_t_minus_1_to_t0_raw",
        "fade_t0_to_t_plus_1_raw",
        "spike_and_fade_raw",
        "marker_definition",
        "figure_annotation_mode",
        "selected_trajectory_version",
        "selected_event_label",
        "selected_legend_label",
        "plot_conference",
        "openalex_career_age_at_pc_year",
        "google_scholar_first_citation_year",
        "google_scholar_career_age_at_pc_year",
        "google_scholar_screenshot_file",
        "google_scholar_screenshot_date",
    ]
    spike_fade_schema_stale = True
    if SPIKE_FADE_CASES_OUT.exists():
        existing_preview = pd.read_csv(SPIKE_FADE_CASES_OUT)
        existing_columns = set(existing_preview.columns)
        spike_fade_schema_stale = not set(spike_fade_required_columns).issubset(
            existing_columns
        )
        if not spike_fade_schema_stale:
            spike_fade_schema_stale = not existing_preview[
                "selected_trajectory_version"
            ].eq("selected_researchers_final_no_story_nodes").all()

    if (
        setup.overwrite_artifacts
        or spike_fade_schema_stale
        or not SPIKE_FADE_CASES_OUT.exists()
    ):
        selected_spike_fade.to_csv(SPIKE_FADE_CASES_OUT, index=False)
        print(f"wrote {SPIKE_FADE_CASES_OUT.relative_to(PROJECT)}")
    else:
        print(f"kept existing {SPIKE_FADE_CASES_OUT.relative_to(PROJECT)}")

    selected_event_study_rows = build_selected_event_study_check_rows(
        selected_spike_fade
    )
    selected_event_study_mean_summary = (
        build_selected_event_study_mean_summary(selected_event_study_rows)
    )
    if (
        setup.overwrite_artifacts
        or not SELECTED_EVENT_STUDY_ANALYSIS_ROWS_OUT.exists()
    ):
        selected_event_study_rows.to_csv(
            SELECTED_EVENT_STUDY_ANALYSIS_ROWS_OUT,
            index=False,
        )
        print(
            "wrote "
            f"{SELECTED_EVENT_STUDY_ANALYSIS_ROWS_OUT.relative_to(PROJECT)}"
        )
    else:
        print(
            "kept existing "
            f"{SELECTED_EVENT_STUDY_ANALYSIS_ROWS_OUT.relative_to(PROJECT)}"
        )

    if (
        setup.overwrite_artifacts
        or not SELECTED_EVENT_STUDY_ANALYSIS_MEAN_SUMMARY_OUT.exists()
    ):
        selected_event_study_mean_summary.to_csv(
            SELECTED_EVENT_STUDY_ANALYSIS_MEAN_SUMMARY_OUT,
            index=False,
        )
        print(
            "wrote "
            f"{SELECTED_EVENT_STUDY_ANALYSIS_MEAN_SUMMARY_OUT.relative_to(PROJECT)}"
        )
    else:
        print(
            "kept existing "
            f"{SELECTED_EVENT_STUDY_ANALYSIS_MEAN_SUMMARY_OUT.relative_to(PROJECT)}"
        )

    refresh_selected_event_study_figure = True
    if (
        refresh_selected_event_study_figure
        or
        setup.overwrite_artifacts
        or not SELECTED_EVENT_STUDY_ANALYSIS_FIGURE_OUT.exists()
    ):
        fig = draw_selected_event_study_check(
            selected_spike_fade,
            selected_event_study_rows,
            selected_event_study_mean_summary,
            balanced_event_rows,
        )
        fig.savefig(
            SELECTED_EVENT_STUDY_ANALYSIS_FIGURE_OUT,
            bbox_inches="tight",
            dpi=STORY_NODE_EXPORT_DPI,
        )
        fig.savefig(
            REPORT_SELECTED_EVENT_STUDY_ANALYSIS_FIGURE_OUT,
            bbox_inches="tight",
            dpi=STORY_NODE_EXPORT_DPI,
        )
        plt.close(fig)
        print(
            "wrote "
            f"{SELECTED_EVENT_STUDY_ANALYSIS_FIGURE_OUT.relative_to(PROJECT)}"
        )
        print(
            "wrote "
            f"{REPORT_SELECTED_EVENT_STUDY_ANALYSIS_FIGURE_OUT.relative_to(PROJECT)}"
        )
    else:
        if not REPORT_SELECTED_EVENT_STUDY_ANALYSIS_FIGURE_OUT.exists():
            REPORT_SELECTED_EVENT_STUDY_ANALYSIS_FIGURE_OUT.write_bytes(
                SELECTED_EVENT_STUDY_ANALYSIS_FIGURE_OUT.read_bytes()
            )
            print(
                "copied "
                f"{REPORT_SELECTED_EVENT_STUDY_ANALYSIS_FIGURE_OUT.relative_to(PROJECT)}"
            )
        print(
            "kept existing "
            f"{SELECTED_EVENT_STUDY_ANALYSIS_FIGURE_OUT.relative_to(PROJECT)}"
        )

    selected_trajectory_sources = build_selected_citation_source_table(
        selected_spike_fade,
        match_selected_conference=True,
    )
    selected_trajectory_t0_sources = selected_trajectory_sources.loc[
        selected_trajectory_sources["event_time"].eq(0)
    ].copy()
    selected_trajectory_t0_summary = (
        build_selected_trajectory_t0_source_summary(
            selected_trajectory_sources
        )
    )
    selected_trajectory_t0_summary = selected_trajectory_t0_summary.merge(
        selected_spike_fade[
            [
                "researcher_id",
                "conference",
                "pc_year",
                "citation_t_0",
                "panel_order",
                "line_order",
            ]
        ],
        on=["researcher_id", "conference", "pc_year"],
        how="left",
        validate="one_to_one",
    ).sort_values(["panel_order", "line_order"])

    source_detail_schema_stale = True
    if SPIKE_FADE_T0_SOURCE_DETAIL_OUT.exists():
        source_detail_columns = set(
            pd.read_csv(SPIKE_FADE_T0_SOURCE_DETAIL_OUT, nrows=0).columns
        )
        source_detail_schema_stale = not {
            "selected_conference",
            "conference",
            "citing_work_id",
            "referenced_work_id",
            "cited_work_title",
        }.issubset(source_detail_columns)

    source_summary_schema_stale = True
    if SPIKE_FADE_T0_SOURCE_SUMMARY_OUT.exists():
        source_summary_columns = set(
            pd.read_csv(SPIKE_FADE_T0_SOURCE_SUMMARY_OUT, nrows=0).columns
        )
        source_summary_schema_stale = not {
            "researcher_id",
            "name",
            "conference",
            "pc_year",
            "citation_t_0",
            "raw_edges",
            "title_collapsed_edges",
            "duplicate_version_edges",
        }.issubset(source_summary_columns)

    if (
        setup.overwrite_artifacts
        or source_detail_schema_stale
        or not SPIKE_FADE_T0_SOURCE_DETAIL_OUT.exists()
    ):
        selected_trajectory_t0_sources.to_csv(
            SPIKE_FADE_T0_SOURCE_DETAIL_OUT,
            index=False,
        )
        print(
            f"wrote {SPIKE_FADE_T0_SOURCE_DETAIL_OUT.relative_to(PROJECT)}"
        )
    else:
        print(
            f"kept existing {SPIKE_FADE_T0_SOURCE_DETAIL_OUT.relative_to(PROJECT)}"
        )

    if (
        setup.overwrite_artifacts
        or source_summary_schema_stale
        or not SPIKE_FADE_T0_SOURCE_SUMMARY_OUT.exists()
    ):
        selected_trajectory_t0_summary.to_csv(
            SPIKE_FADE_T0_SOURCE_SUMMARY_OUT,
            index=False,
        )
        print(
            f"wrote {SPIKE_FADE_T0_SOURCE_SUMMARY_OUT.relative_to(PROJECT)}"
        )
    else:
        print(
            f"kept existing {SPIKE_FADE_T0_SOURCE_SUMMARY_OUT.relative_to(PROJECT)}"
        )

    high_res_node_paths = build_high_res_story_node_images()
    story_node_quality_stale = True
    existing_figure_paths = [
        path
        for path in [SPIKE_FADE_FIGURE_OUT, REPORT_SPIKE_FADE_FIGURE_OUT]
        if path.exists()
    ]
    if high_res_node_paths and len(existing_figure_paths) == 2:
        figure_mtime = min(path.stat().st_mtime for path in existing_figure_paths)
        story_node_quality_stale = any(
            path.stat().st_mtime > figure_mtime for path in high_res_node_paths
        )

    if (
        setup.overwrite_artifacts
        or spike_fade_schema_stale
        or story_node_quality_stale
        or not SPIKE_FADE_FIGURE_OUT.exists()
    ):
        fig = draw_spike_fade_examples(selected_spike_fade)
        fig.savefig(
            SPIKE_FADE_FIGURE_OUT,
            bbox_inches="tight",
            dpi=STORY_NODE_EXPORT_DPI,
        )
        fig.savefig(
            REPORT_SPIKE_FADE_FIGURE_OUT,
            bbox_inches="tight",
            dpi=STORY_NODE_EXPORT_DPI,
        )
        plt.close(fig)
        print(f"wrote {SPIKE_FADE_FIGURE_OUT.relative_to(PROJECT)}")
        print(f"wrote {REPORT_SPIKE_FADE_FIGURE_OUT.relative_to(PROJECT)}")
    else:
        if not REPORT_SPIKE_FADE_FIGURE_OUT.exists():
            REPORT_SPIKE_FADE_FIGURE_OUT.write_bytes(
                SPIKE_FADE_FIGURE_OUT.read_bytes()
            )
            print(
                f"copied {REPORT_SPIKE_FADE_FIGURE_OUT.relative_to(PROJECT)}"
            )
        print(f"kept existing {SPIKE_FADE_FIGURE_OUT.relative_to(PROJECT)}")

    required_gs_screenshots = [
        GS_SCREENSHOT_DIR / filename
        for filename in GOOGLE_SCHOLAR_SCREENSHOT_FILES.values()
    ]
    missing_gs_screenshots = [
        path for path in required_gs_screenshots if not path.exists()
    ]
    if missing_gs_screenshots:
        print(
            "Skipping Google Scholar screenshot grid/copy; "
            "selected screenshot assets are not included in this copy."
        )
    else:
        expected_gs_grid_size = (
            5 * 1250 + 6 * 45,
            2 * (540 + 64) + 3 * 45,
        )
        gs_grid_stale = True
        if REPORT_GS_SCREENSHOT_GRID_OUT.exists():
            gs_grid_stale = (
                Image.open(REPORT_GS_SCREENSHOT_GRID_OUT).size
                != expected_gs_grid_size
            )

        if setup.overwrite_artifacts or gs_grid_stale or spike_fade_schema_stale:
            gs_grid = draw_google_scholar_screenshot_grid()
            gs_grid.save(REPORT_GS_SCREENSHOT_GRID_OUT)
            print(f"wrote {REPORT_GS_SCREENSHOT_GRID_OUT.relative_to(PROJECT)}")
        else:
            print(
                f"kept existing {REPORT_GS_SCREENSHOT_GRID_OUT.relative_to(PROJECT)}"
            )

        copied_screenshots = copy_google_scholar_screenshots_for_latex()
        print(
            "prepared "
            f"{len(copied_screenshots)} Google Scholar screenshots for LaTeX"
        )

    display(selected_spike_fade)


isolated balanced event units: 403
raw spike-and-fade cases: 100 (24.8%)
positive PC-year jump and follow-up mean below baseline: 67 (16.6%)


,conference,sum,count,share
0,ICFP,21,106,0.198113
1,OOPSLA,8,21,0.380952
2,PLDI,37,138,0.268116
3,POPL,34,138,0.246377


wrote step_4_artifacts/summary_tables/selected_spike_and_fade_cases.csv
wrote step_4_artifacts/summary_tables/selected_pc_event_study_analysis_rows.csv
wrote step_4_artifacts/summary_tables/selected_event_means.csv


/var/folders/02/q9zpk4h12sx3mq390472dklr0000gn/T/ipykernel_3283/763219323.py:909: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


wrote step_4_artifacts/main_text_figures/selected_pc_event_study_analysis.pdf
wrote report_latex/figures/selected_pc_event_study_analysis.pdf
wrote step_4_artifacts/summary_tables/selected_t0_source_detail.csv
wrote step_4_artifacts/summary_tables/selected_t0_source_summary.csv
wrote step_4_artifacts/main_text_figures/selected_pc_trajectories.pdf
wrote report_latex/figures/selected_pc_trajectories.pdf
Skipping Google Scholar screenshot grid/copy; selected screenshot assets are not included in this copy.


,selection_rank,panel_order,line_order,researcher_id,name,conference,plot_conference,pc_year,selected_event_label,selected_trajectory_version,...,google_scholar_career_age_at_pc_year,google_scholar_screenshot_file,selected_legend_label,baseline_raw_citations,pc_year_jump_raw,jump_t_minus_1_to_t0_raw,fade_t0_to_t_plus_1_raw,spike_and_fade_raw,marker_definition,figure_annotation_mode
0,1,1,1,clementpitclaudel,Clément Pit-Claudel,PLDI,PLDI,2024,PLDI 2024,selected_researchers_final_no_story_nodes,...,8,Clement_.png,PLDI 2024,4.5,8.5,10.0,-8.0,True,trajectory_only,multi_researcher_selected_trajectories
1,1,1,2,clementpitclaudel,Clément Pit-Claudel,OOPSLA_COMBINED,OOPSLA combined,2025,OOPSLA 2025,selected_researchers_final_no_story_nodes,...,9,Clement_.png,OOPSLA 2025,2.0,7.0,7.0,NaN,False,trajectory_only,multi_researcher_selected_trajectories
2,2,2,1,viktorkuncak,Viktor Kunčak,OOPSLA,OOPSLA,2020,OOPSLA 2020,selected_researchers_final_no_story_nodes,...,17,victor_kuncak.png,OOPSLA 2020,6.5,13.5,12.0,-10.0,True,trajectory_only,multi_researcher_selected_trajectories
3,2,2,2,viktorkuncak,Viktor Kunčak,PLDI,PLDI,2021,PLDI 2021,selected_researchers_final_no_story_nodes,...,18,victor_kuncak.png,PLDI 2021,9.0,9.0,11.0,-13.0,True,trajectory_only,multi_researcher_selected_trajectories
4,3,3,1,peterthiemann,Peter Thiemann,ICFP,ICFP,2023,ICFP 2023,selected_researchers_final_no_story_nodes,...,28,Peter_thieman.png,ICFP 2023,2.5,5.5,6.0,-7.0,True,trajectory_only,multi_researcher_selected_trajectories
5,4,4,1,simonjgay,Simon J. Gay,ICFP,ICFP,2023,ICFP 2023,selected_researchers_final_no_story_nodes,...,29,simon_j_gay.png,ICFP 2023,2.5,8.5,8.0,-11.0,True,trajectory_only,multi_researcher_selected_trajectories
6,5,5,1,martinelsman,Martin Elsman,ICFP,ICFP,2024,ICFP 2024,selected_researchers_final_no_story_nodes,...,27,Martin_Elsman_career_age_ICFP.png,ICFP 2024,0.0,8.0,8.0,-7.0,True,trajectory_only,multi_researcher_selected_trajectories
7,6,6,1,ezgicicek,Ezgi Çiçek,ICFP,ICFP,2019,ICFP 2019,selected_researchers_final_no_story_nodes,...,5,Ezgi_Çiçek_Career_Age_ICFP.png,ICFP 2019,3.0,4.0,2.0,-5.0,True,trajectory_only,multi_researcher_selected_trajectories
8,7,7,1,brandonlucia,Brandon Lucia,OOPSLA,OOPSLA,2019,OOPSLA 2019,selected_researchers_final_no_story_nodes,...,10,Brandon Lucia.png,OOPSLA 2019,5.5,18.5,23.0,-10.0,True,trajectory_only,multi_researcher_selected_trajectories
9,8,8,1,jonathanbrachthauser,Jonathan Immanuel Brachthäuser,POPL,POPL,2024,POPL 2024,selected_researchers_final_no_story_nodes,...,10,Jonathan Immanuel Brachthäuser .png,POPL 2024,0.5,15.5,15.0,-14.0,True,trajectory_only,multi_researcher_selected_trajectories


## Outputs and figure previews

This notebook always creates the reliable author target list. The
author metrics table and the event study outputs are created after
OpenAlex author metadata are available. The event study uses
`career_age_publication_year`, which is the first OpenAlex
Computer Science publication year when that year is 1980 or later.

Main outputs:

- `step_4_artifacts/summary_tables/career_age_reliable_author_list.csv`
- `step_4_data/prepared/researcher_openalex_author_metrics.parquet`
- `step_4_data/prepared/career_age_event_window_rows.parquet`
- `step_4_artifacts/summary_tables/career_age_event_study_log_summary.csv`
- `step_4_artifacts/summary_tables/career_age_event_study_raw_summary.csv`
- `step_4_artifacts/summary_tables/career_age_log_by_conf.csv`
- `step_4_artifacts/summary_tables/career_age_raw_by_conf.csv`
- `step_4_artifacts/summary_tables/icfp_selection_log.csv`
- `step_4_artifacts/summary_tables/icfp_selection_raw.csv`
- `step_4_artifacts/summary_tables/icfp_selected_jumps.csv`
- `step_4_artifacts/summary_tables/icfp_selected_sources.csv`
- `step_4_artifacts/summary_tables/icfp_selected_prior_summary.csv`
- `step_4_artifacts/summary_tables/icfp_selected_prior_work.csv`
- `step_4_artifacts/summary_tables/icfp_selected_citing_papers.csv`
- `step_4_artifacts/summary_tables/selected_spike_and_fade_cases.csv`
- `step_4_artifacts/summary_tables/selected_pc_event_study_analysis_rows.csv`
- `step_4_artifacts/summary_tables/selected_event_means.csv`
- `step_4_artifacts/summary_tables/selected_t0_source_summary.csv`
- `step_4_artifacts/summary_tables/selected_t0_source_detail.csv`
Figures produced by this notebook are listed and previewed below.

- `step_4_artifacts/figures_pre_pc_mean_baseline_event_study/popl_fixed_buckets.pdf`
- `step_4_artifacts/figures_pre_pc_mean_baseline_event_study/icfp_fixed_buckets.pdf`
- `step_4_artifacts/figures_pre_pc_mean_baseline_event_study/oopsla_fixed_buckets.pdf`
- `step_4_artifacts/figures_pre_pc_mean_baseline_event_study/pldi_before_2021_fixed_buckets.pdf`
- `step_4_artifacts/figures_pre_pc_mean_baseline_event_study/pldi_2021_onward_fixed_buckets.pdf`
- `step_4_artifacts/figures_pre_pc_mean_baseline_event_study/icfp_log.pdf`
- `step_4_artifacts/figures_pre_pc_mean_baseline_event_study/icfp_raw.pdf`
- `step_4_artifacts/main_text_figures/selected_pc_event_study_analysis.pdf`
- `step_4_artifacts/main_text_figures/selected_pc_trajectories.pdf`

Each narrative PDF has three pages: all isolated balanced event units,
no earlier main PC evidence found, and no earlier service. The default
fixed bucket PDFs use 0-9, 10-19, and
20+ years. PLDI is split into `pldi_before_2021` and
`pldi_2021_onward` because the public committee label changes around
2021.

In [10]:
figure_outputs = [
    ("POPL fixed career-age buckets", PRE_PC_MEAN_BASELINE_FIGURES / "popl_fixed_buckets.pdf"),
    ("ICFP fixed career-age buckets", PRE_PC_MEAN_BASELINE_FIGURES / "icfp_fixed_buckets.pdf"),
    ("OOPSLA fixed career-age buckets", PRE_PC_MEAN_BASELINE_FIGURES / "oopsla_fixed_buckets.pdf"),
    ("PLDI before 2021 fixed career-age buckets", PRE_PC_MEAN_BASELINE_FIGURES / "pldi_before_2021_fixed_buckets.pdf"),
    ("PLDI 2021 onward fixed career-age buckets", PRE_PC_MEAN_BASELINE_FIGURES / "pldi_2021_onward_fixed_buckets.pdf"),
    ("ICFP main-text log-scale robustness figure", PRE_PC_MEAN_BASELINE_FIGURES / "icfp_log.pdf"),
    ("ICFP main-text raw-count robustness figure", PRE_PC_MEAN_BASELINE_FIGURES / "icfp_raw.pdf"),
    ("Selected PC-member event-study analysis", SELECTED_EVENT_STUDY_ANALYSIS_FIGURE_OUT),
    ("Selected PC-member citation trajectories", SPIKE_FADE_FIGURE_OUT),
]

figure_table = pd.DataFrame(
    {
        "figure": label,
        "path": str(path.relative_to(PROJECT)),
        "exists": path.exists(),
    }
    for label, path in figure_outputs
)
display(figure_table)

notebook_root = Path("..") / ".."

def show_pdf(label, path, height=620):
    rel_project_path = path.relative_to(PROJECT)
    display(Markdown(f"**{label}**  \n`{rel_project_path}`"))
    if path.exists():
        display(IFrame(src=str(notebook_root / rel_project_path), width="100%", height=height))
    else:
        display(Markdown("Missing figure file."))

for label, path in figure_outputs:
    show_pdf(label, path)

,figure,path,exists
0,POPL fixed career-age buckets,step_4_artifacts/figures_pre_pc_mean_baseline_...,True
1,ICFP fixed career-age buckets,step_4_artifacts/figures_pre_pc_mean_baseline_...,True
2,OOPSLA fixed career-age buckets,step_4_artifacts/figures_pre_pc_mean_baseline_...,True
3,PLDI before 2021 fixed career-age buckets,step_4_artifacts/figures_pre_pc_mean_baseline_...,True
4,PLDI 2021 onward fixed career-age buckets,step_4_artifacts/figures_pre_pc_mean_baseline_...,True
5,ICFP main-text log-scale robustness figure,step_4_artifacts/figures_pre_pc_mean_baseline_...,True
6,ICFP main-text raw-count robustness figure,step_4_artifacts/figures_pre_pc_mean_baseline_...,True
7,Selected PC-member event-study analysis,step_4_artifacts/main_text_figures/selected_pc...,True
8,Selected PC-member citation trajectories,step_4_artifacts/main_text_figures/selected_pc...,True


**POPL fixed career-age buckets**  
`step_4_artifacts/figures_pre_pc_mean_baseline_event_study/popl_fixed_buckets.pdf`

**ICFP fixed career-age buckets**  
`step_4_artifacts/figures_pre_pc_mean_baseline_event_study/icfp_fixed_buckets.pdf`

**OOPSLA fixed career-age buckets**  
`step_4_artifacts/figures_pre_pc_mean_baseline_event_study/oopsla_fixed_buckets.pdf`

**PLDI before 2021 fixed career-age buckets**  
`step_4_artifacts/figures_pre_pc_mean_baseline_event_study/pldi_before_2021_fixed_buckets.pdf`

**PLDI 2021 onward fixed career-age buckets**  
`step_4_artifacts/figures_pre_pc_mean_baseline_event_study/pldi_2021_onward_fixed_buckets.pdf`

**ICFP main-text log-scale robustness figure**  
`step_4_artifacts/figures_pre_pc_mean_baseline_event_study/icfp_log.pdf`

**ICFP main-text raw-count robustness figure**  
`step_4_artifacts/figures_pre_pc_mean_baseline_event_study/icfp_raw.pdf`

**Selected PC-member event-study analysis**  
`step_4_artifacts/main_text_figures/selected_pc_event_study_analysis.pdf`

**Selected PC-member citation trajectories**  
`step_4_artifacts/main_text_figures/selected_pc_trajectories.pdf`